# 05 - Data Preprocessing

This notebook prepares the ACDC coworking membership dataset for clustering.

Preprocessing decisions follow the findings from the Data Understanding notebook, with the goal of creating reliable and comparable features for customer segmentation.

## 1. Setup

Import the necessary libraries and configure display settings for data preprocessing and analysis.

In [164]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid")

## 2. Data Loading

Load the original dataset and create a working copy for preprocessing.

In [165]:
# Define project paths
BASE_PATH = Path.cwd().parent
DATA = BASE_PATH / "data"
RAW = DATA / "raw"
DATASET = RAW / "acdc_dataset.csv"

# Load the dataset
df_raw = pd.read_csv(DATASET, sep=";")
df = df_raw.copy()

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

Dataset shape: 15,465 rows × 31 columns


,member_id,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,last_badge_swipe,days_since_last_visit,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period,meeting_room_hours,event_space_hours,guest_passes_issued,uses_meeting_rooms,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
0,CW000001,38.00,Female,Information Technology,Portugal,False,0,flex,3,2021-08-01,2023-03-06,2023-03-23,-17,220,2.65,26.09,22,0.00,0.00,18.00,False,False,True,0,False,True,True,False,False,2,4477.97
1,CW000002,38.00,Male,Information Technology,Portugal,True,2,HOT_DESK,2,2023-11-24,1970-01-01,2023-11-24,37,2,0.32,10.68,1,0.00,0.00,0.00,False,False,False,0,True,False,False,False,False,0,165.86
2,CW000003,43.00,Male,Education,DE,True,1,flex,3,2021-12-03,1970-01-01,2023-07-22,163,19,0.17,26.09,2,0.00,0.00,0.00,False,False,False,0,False,False,False,False,False,1,4486.17
3,CW000004,38.00,F,Legal,portugal,False,0,flex,3,2022-01-02,2022-02-15,2022-01-02,44,8,1.23,18.74,9,0.00,0.00,0.00,False,False,False,0,False,False,False,False,False,0,258.63
4,CW000005,45.00,Female,Information Technology,United Kingdom,False,0,hot_desk,2,2023-10-24,1970-01-01,2023-12-17,15,4,0.38,17.39,3,0.00,0.00,0.00,False,False,False,0,False,False,True,False,False,0,335.74


## 3. Data Quality and Cleaning

Apply the cleaning decisions identified during EDA to improve data quality before clustering.

### 3.1. Initial Quality Check

Check duplicates, missing values and data types to confirm the issues requiring treatment.

In [166]:
# Check duplicates
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate member IDs: {df['member_id'].duplicated().sum():,}")

# Check missing values
missing = df.isna().sum()
display(missing[missing > 0].sort_values(ascending=False).to_frame("Missing Values"))

# Check data types
display(df.dtypes.to_frame("Data Type"))

Duplicate rows: 15
Duplicate member IDs: 15


,Missing Values
meeting_room_hours,1298
guest_passes_issued,1271
lifetime_value,1040
age,558
gender,530
country,430
plan_tier,105


,Data Type
member_id,object
age,float64
gender,object
industry,object
country,object
is_team_account,bool
n_colleagues_on_account,int64
plan_tier,object
plan_days_per_week,int64
membership_start,object


### 3.2. Duplicate Removal

Remove the 15 duplicate records to ensure each member is represented only once during clustering.

In [167]:
# Remove exact duplicates
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)

print(f"Rows removed: {rows_before - len(df):,}")
print(f"Remaining rows: {len(df):,}")
print(f"Duplicate member IDs remaining: {df['member_id'].duplicated().sum():,}")

Rows removed: 15
Remaining rows: 15,450
Duplicate member IDs remaining: 0


### 3.3. Invalid Values and Inconsistencies

Identify invalid values and placeholder codes reported during EDA before applying corrections.

In [168]:
# Check invalid values identified during EDA
checks = {
    "Age outside 16–100": ~df["age"].between(16, 100) & df["age"].notna(),
    "Plan days outside 0–7": ~df["plan_days_per_week"].between(0, 7),
    "Plan days equal to 0": df["plan_days_per_week"] == 0,
    "Negative colleagues": df["n_colleagues_on_account"] < 0,
    "Negative total visits": df["total_visits"] < 0,
    "Negative renewals": df["n_renewals"] < 0,
    "Negative meeting room hours": df["meeting_room_hours"] < 0,
    "Negative event space hours": df["event_space_hours"] < 0,
    "Negative guest passes": df["guest_passes_issued"] < 0,
    "Recency placeholder (999)": df["days_since_last_visit"] == 999,
    "Negative recency": df["days_since_last_visit"] < 0,
    "Membership end placeholder": df["membership_end"] == "1970-01-01"
}

invalid_report = pd.DataFrame({
    "Affected Rows": {name: mask.sum() for name, mask in checks.items()}
})

display(invalid_report)

,Affected Rows
Age outside 16–100,143
Plan days outside 0–7,54
Plan days equal to 0,43
Negative colleagues,565
Negative total visits,42
Negative renewals,66
Negative meeting room hours,16
Negative event space hours,8
Negative guest passes,7
Recency placeholder (999),1298


### 3.4. Invalid Numeric Values

Replace impossible numeric values with NaN, following the EDA findings. Missing values will be handled later.

In [169]:
# Define valid ranges based on EDA
valid_ranges = {
    "age": df["age"].between(16, 100),
    "plan_days_per_week": df["plan_days_per_week"].between(1, 7),
    "n_colleagues_on_account": df["n_colleagues_on_account"] >= 0,
    "total_visits": df["total_visits"] >= 0,
    "n_renewals": df["n_renewals"] >= 0
}

# Replace invalid values with NaN
for col, valid in valid_ranges.items():
    df.loc[~valid, col] = np.nan

# Verify missing values after corrections
display(df[list(valid_ranges)].isna().sum().to_frame("Missing Values"))

,Missing Values
age,700
plan_days_per_week,97
n_colleagues_on_account,565
total_visits,42
n_renewals,66


### 3.5. Service-Use Corrections

Use service flags to replace invalid or missing quantities with zero when the service was not used. Otherwise, preserve missing values for later imputation.

In [170]:
# Match service quantities with their usage flags
service_pairs = {
    "meeting_room_hours": "uses_meeting_rooms",
    "event_space_hours": "uses_event_space",
    "guest_passes_issued": "uses_guest_passes"
}

# Correct invalid or missing service quantities
for quantity, flag in service_pairs.items():
    invalid = df[quantity].isna() | (df[quantity] < 0)
    not_used = df[flag] == False

    # No service usage means zero consumption
    df.loc[invalid & not_used, quantity] = 0

    # Otherwise, the quantity remains unknown
    df.loc[invalid & ~not_used, quantity] = np.nan

# Verify remaining issues
service_report = pd.DataFrame({
    "Missing Values": df[list(service_pairs)].isna().sum(),
    "Negative Values": (df[list(service_pairs)] < 0).sum()
})

display(service_report)

,Missing Values,Negative Values
meeting_room_hours,409,0
event_space_hours,8,0
guest_passes_issued,210,0


### 3.6. Date and Recency Corrections

Convert date columns to datetime and address the placeholder dates and temporal inconsistencies identified during EDA.

In [171]:
# Convert date columns to datetime
date_cols = ["membership_start", "membership_end", "last_badge_swipe"]

for col in date_cols:
    original = df[col].copy()
    df[col] = pd.to_datetime(df[col], format="mixed", dayfirst=True, errors="coerce")

    failed = (original.notna() & df[col].isna()).sum()
    print(f"{col}: {failed} unparsed values")

# Verify date ranges
display(df[date_cols].agg(["min", "max"]))

membership_start: 0 unparsed values
membership_end: 0 unparsed values
last_badge_swipe: 0 unparsed values


,membership_start,membership_end,last_badge_swipe
min,2021-01-01,1970-01-01,2020-09-06
max,2023-12-30,2024-01-01,2024-04-28


#### 3.6.1. Active Memberships

The date 1970-01-01 represents memberships without a recorded end date. Preserve this information using an is_active flag and replace the placeholder with NaT.

In [172]:
# Identify active memberships before removing placeholders
active_mask = df["membership_end"] == pd.Timestamp("1970-01-01")

df["is_active"] = active_mask.astype(int)
df.loc[active_mask, "membership_end"] = pd.NaT

print(f"Active memberships: {df['is_active'].sum():,}")
print(f"Missing membership end dates: {df['membership_end'].isna().sum():,}")

Active memberships: 3,954
Missing membership end dates: 3,954


#### 3.6.2. Temporal Consistency

Flag chronological inconsistencies identified during EDA. Preserve these records for further investigation before deciding whether corrections are necessary.

In [173]:
# Check temporal consistency
temporal_checks = {
    "start_after_end": df["membership_start"] > df["membership_end"],
    "swipe_before_start": df["last_badge_swipe"] < df["membership_start"],
    "swipe_after_end": df["last_badge_swipe"] > df["membership_end"]
}

# Create flags without removing observations
for name, mask in temporal_checks.items():
    df[f"flag_{name}"] = mask.fillna(False)

# Summarize inconsistencies
temporal_report = pd.DataFrame({
    "Affected Rows": {
        name: int(mask.sum()) for name, mask in temporal_checks.items()
    }
})

display(temporal_report)

,Affected Rows
start_after_end,152
swipe_before_start,94
swipe_after_end,213


#### 3.6.3. Recency Values

Replace the 999 missing-value code with NaN and investigate negative recency values using membership dates.

In [174]:
# Replace the recency missing-value code
placeholder_mask = df["days_since_last_visit"] == 999
df.loc[placeholder_mask, "days_since_last_visit"] = np.nan

# Reconstruct recency using membership dates
reference_date = pd.Timestamp("2024-01-01")
effective_end = df["membership_end"].fillna(reference_date)
expected_recency = (effective_end - df["last_badge_swipe"]).dt.days

# Compare recorded and calculated recency
valid = df["days_since_last_visit"].notna() & expected_recency.notna()
difference = df["days_since_last_visit"] - expected_recency

print(f"Recency placeholders removed: {placeholder_mask.sum():,}")
print(f"Exact date matches: {(difference[valid] == 0).sum():,}")
print(f"Matches within 1 day: {(difference[valid].abs() <= 1).sum():,}")
print(f"Differences above 1 day: {(difference[valid].abs() > 1).sum():,}")

# Inspect negative recency values
negative_recency = df["days_since_last_visit"] < 0
print(f"Negative recency values: {negative_recency.sum():,}")

display(df.loc[negative_recency, [
    "member_id",
    "is_active",
    "membership_end",
    "last_badge_swipe",
    "days_since_last_visit",
    "flag_swipe_after_end"
]].head(10))

Recency placeholders removed: 1,298
Exact date matches: 13,195
Matches within 1 day: 13,867
Differences above 1 day: 285
Negative recency values: 56


,member_id,is_active,membership_end,last_badge_swipe,days_since_last_visit,flag_swipe_after_end
0,CW000001,0,2023-03-06,2023-03-23,-17.00,True
493,CW000480,0,2021-09-15,2022-05-13,-240.00,True
511,CW000498,0,2022-02-10,2023-11-15,-643.00,True
557,CW000541,0,2022-08-30,2022-12-10,-102.00,True
811,CW000788,1,NaT,2024-04-11,-101.00,False
1047,CW001018,0,2021-12-17,2023-12-21,-734.00,True
1422,CW001384,0,2023-11-20,2023-12-28,-38.00,True
1513,CW001474,1,NaT,2024-03-29,-88.00,False
1522,CW001483,1,NaT,2024-04-13,-103.00,False
1549,CW001509,0,2023-10-11,2023-12-16,-67.00,True


#### 3.6.4. Recency Validation

Investigate recency inconsistencies using membership dates and identify values requiring correction.

In [175]:
# Identify recency inconsistencies
df["flag_recency_mismatch"] = valid & (difference.abs() > 1)
df["flag_negative_recency"] = df["days_since_last_visit"] < 0
df["flag_swipe_after_reference"] = df["last_badge_swipe"] > reference_date

# Summarize the issues
recency_report = {
    "Recency mismatches": df["flag_recency_mismatch"].sum(),
    "Negative recency": df["flag_negative_recency"].sum(),
    "Swipes after reference date": df["flag_swipe_after_reference"].sum(),
    "Negative recency with late swipe": (
        df["flag_negative_recency"] &
        (df["flag_swipe_after_end"] | df["flag_swipe_after_reference"])
    ).sum()
}

display(pd.Series(recency_report, name="Affected Rows").to_frame())

# Inspect mismatched observations
mismatch = df["flag_recency_mismatch"]

examples = df.loc[mismatch, [
    "member_id",
    "is_active",
    "membership_end",
    "last_badge_swipe",
    "days_since_last_visit"
]].copy()

examples["calculated_recency"] = expected_recency[mismatch]

display(examples.head(10))

,Affected Rows
Recency mismatches,285
Negative recency,56
Swipes after reference date,34
Negative recency with late swipe,56


,member_id,is_active,membership_end,last_badge_swipe,days_since_last_visit,calculated_recency
165,CW000161,0,2021-10-17,2023-02-12,5.00,-483
290,CW000284,0,2021-03-12,2022-08-11,6.00,-517
307,CW000301,0,2022-08-16,2022-11-20,4.00,-96
330,CW000323,1,NaT,2024-03-19,146.00,-78
466,CW000454,1,NaT,2021-07-14,149.00,901
535,CW015353,0,2023-06-02,2023-05-30,8.00,3
556,CW015291,0,2022-04-07,2021-12-04,8.00,124
601,CW000585,0,2022-08-19,2023-07-17,5.00,-332
679,CW000660,0,2023-12-07,2020-12-12,0.00,1090
723,CW000703,0,2023-05-03,2023-05-15,2.00,-12


#### 3.6.5. Recency Corrections

Replace negative recency values with NaN. Preserve other mismatches for further evaluation, as the recorded dates may also contain errors.

In [176]:
# Preserve negative recency information in the existing flag
negative_count = df["flag_negative_recency"].sum()

# Replace negative recency values with NaN
df.loc[df["flag_negative_recency"], "days_since_last_visit"] = np.nan

# Verify the corrections
print(f"Negative recency values corrected: {negative_count:,}")
print(f"Remaining negative values: {(df['days_since_last_visit'] < 0).sum():,}")
print(f"Missing recency values: {df['days_since_last_visit'].isna().sum():,}")
print(f"Recency mismatches flagged for review: {df['flag_recency_mismatch'].sum():,}")

Negative recency values corrected: 56
Remaining negative values: 0
Missing recency values: 1,354
Recency mismatches flagged for review: 285


#### 3.6.6. Swipe Reliability
Flag late badge swipes using the provisional 30-day tolerance to avoid reconstructing recency from unreliable dates.

In [177]:
# Define the tolerance for late swipes
GRACE_PERIOD = 30

# Calculate days after membership expiration
days_after_end = (
    df["last_badge_swipe"] - df["membership_end"]
).dt.days

# Flag unreliable swipe dates
df["flag_unreliable_swipe"] = (
    df["flag_swipe_before_start"] |
    (days_after_end > GRACE_PERIOD) |
    df["flag_swipe_after_reference"]
)

# Summarize the results
print(f"Unreliable swipe records: {df['flag_unreliable_swipe'].sum():,}")
print(f"Reliable swipe records: {(~df['flag_unreliable_swipe']).sum():,}")

Unreliable swipe records: 295
Reliable swipe records: 15,155


### 3.7. Categorical Cleaning

Identify and standardize inconsistent category labels to prevent equivalent values from being treated as different categories.

In [178]:
# Inspect categorical values
categorical_cols = ["gender", "industry", "country", "plan_tier"]

for col in categorical_cols:
    print(f"\n{col} ({df[col].nunique(dropna=True)} unique categories)")
    display(df[col].value_counts(dropna=False).to_frame("Count"))


gender (12 unique categories)


,Count
gender,
Female,7405
Male,6166
NaN,530
Other,465
female,169
F,143
Female,141
M,138
male,132



industry (43 unique categories)


,Count
industry,
Information Technology,3320
Consulting,1893
Marketing & PR,1834
Design & Creative,1821
Finance,1134
Other,1100
Legal,807
Education,727
Unknown,638



country (33 unique categories)


,Count
country,
Portugal,9950
United Kingdom,1104
Germany,850
Spain,711
France,506
Netherlands,463
NaN,429
Other (Europe),429
Ireland,232



plan_tier (19 unique categories)


,Count
plan_tier,
flex,5179
hot_desk,4971
full_time,2320
dedicated_desk,1478
private_office,629
Flex,138
FLEX,132
NaN,105
HOT_DESK,82


In [179]:
# Define category mappings
category_mappings = {
    "gender": {
        "female": "Female", "f": "Female",
        "male": "Male", "m": "Male",
        "other": "Other", "o": "Other"
    },
    "industry": {
        "information technology": "Information Technology",
        "it": "Information Technology",
        "tech": "Information Technology",
        "consulting": "Consulting",
        "marketing & pr": "Marketing & PR",
        "marketing/pr": "Marketing & PR",
        "marketing and pr": "Marketing & PR",
        "design & creative": "Design & Creative",
        "design/creative": "Design & Creative",
        "design and creative": "Design & Creative",
        "finance": "Finance",
        "legal": "Legal",
        "education": "Education",
        "health": "Health",
        "real estate": "Real Estate",
        "realestate": "Real Estate",
        "real-estate": "Real Estate",
        "other": "Other",
        "unknown": "Unknown"
    },
    "country": {
        "portugal": "Portugal",
        "portucal": "Portugal",
        "pt": "Portugal",
        "united kingdom": "United Kingdom",
        "uk": "United Kingdom",
        "u.k.": "United Kingdom",
        "gb": "United Kingdom",
        "germany": "Germany",
        "de": "Germany",
        "deutschland": "Germany",
        "spain": "Spain",
        "es": "Spain",
        "espana": "Spain",
        "españa": "Spain",
        "france": "France",
        "fr": "France",
        "netherlands": "Netherlands",
        "the netherlands": "Netherlands",
        "holland": "Netherlands",
        "nl": "Netherlands",
        "ireland": "Ireland",
        "ie": "Ireland",
        "other (europe)": "Other (Europe)"
    },
    "plan_tier": {
        "flex": "flex",
        "hot_desk": "hot_desk",
        "hot desk": "hot_desk",
        "hotdesk": "hot_desk",
        "full_time": "full_time",
        "full time": "full_time",
        "fulltime": "full_time",
        "dedicated_desk": "dedicated_desk",
        "dedicated desk": "dedicated_desk",
        "dedicateddesk": "dedicated_desk",
        "private_office": "private_office",
        "private office": "private_office",
        "privateoffice": "private_office"
    }
}

# Standardize categorical values
for col, mapping in category_mappings.items():
    normalized = df[col].astype("string").str.strip().str.lower()

    # Check for unexpected categories before mapping
    unmapped = normalized.notna() & ~normalized.isin(mapping)
    if unmapped.any():
        raise ValueError(f"Unmapped values in {col}: {normalized[unmapped].unique().tolist()}")

    df[col] = normalized.map(mapping)

# Verify standardized categories
for col in category_mappings:
    print(f"\n{col}: {df[col].nunique()} categories")
    display(df[col].value_counts(dropna=False).to_frame("Count"))


gender: 3 categories


,Count
gender,
Female,7858
Male,6565
NaN,530
Other,497



industry: 11 categories


,Count
industry,
Information Technology,3557
Consulting,2055
Design & Creative,1974
Marketing & PR,1962
Finance,1219
Other,1175
Legal,878
Education,775
Unknown,693



country: 8 categories


,Count
country,
Portugal,10499
United Kingdom,1165
Germany,901
Spain,741
France,533
Netherlands,487
Other (Europe),451
NaN,429
Ireland,244



plan_tier: 5 categories


,Count
plan_tier,
flex,5449
hot_desk,5212
full_time,2471
dedicated_desk,1549
private_office,664
NaN,105


### 3.8. Lifetime Value Cleaning
Convert monetary values to numeric and implement targeted scaling corrections. The EDA already describes the distributions and notes that negative amounts may reflect refunds. Original financial values and correction flags are retained.

In [180]:
# Convert monetary values to numeric, preserving original source values in df_raw
missing_before = df["lifetime_value"].isna().sum()
df["lifetime_value"] = (
    df["lifetime_value"].astype("string")
    .str.replace(r"[€$â‚¬\s]", "", regex=True)
    .str.replace(r",(?=\d{3}(?:[,.]|$))", "", regex=True)
    .str.replace(",", ".", regex=False)
)
df["lifetime_value"] = pd.to_numeric(df["lifetime_value"], errors="coerce")
print(f"Conversion failures: {df['lifetime_value'].isna().sum() - missing_before:,}")
print(f"Missing values: {df['lifetime_value'].isna().sum():,}")
print(f"Zero values: {df['lifetime_value'].eq(0).sum():,}")
print(f"Negative values: {df['lifetime_value'].lt(0).sum():,}")

Conversion failures: 0
Missing values: 1,040
Zero values: 1,742
Negative values: 82


#### 3.8.1. Membership Duration for Revenue Validation
Calculate membership duration in months using the reference date for open-ended memberships. Only positive durations will be used to assess monthly revenue plausibility.

In [181]:
# Derive membership duration for revenue plausibility checks
membership_days = (df["membership_end"].fillna(reference_date) - df["membership_start"]).dt.days
membership_months = (membership_days / 30.44).where(membership_days > 0)
print(f"Memberships with positive duration: {membership_months.notna().sum():,}")

Memberships with positive duration: 15,298


#### 3.8.2. Plan-Based Monthly Revenue Benchmarks
Use the median and 95th percentile of plausible positive LTV per membership month as benchmarks for identifying potential scaling errors.

In [182]:
# Estimate reference monthly revenue using plausible observed values
regular_mask = df["lifetime_value"].between(0, 100000, inclusive="neither") & membership_months.ge(1)
regular_ltv = df.loc[regular_mask, ["plan_tier", "lifetime_value"]].copy()
regular_ltv["monthly_value"] = regular_ltv["lifetime_value"] / membership_months.loc[regular_mask]
monthly_benchmarks = regular_ltv.groupby("plan_tier")["monthly_value"].agg(
    count="count",
    median="median",
    p95=lambda x: x.quantile(0.95)
)
display(monthly_benchmarks.round(2))

,count,median,p95
plan_tier,,,
dedicated_desk,1164,407.03,537.03
flex,4177,204.83,454.76
full_time,1828,312.67,573.98
hot_desk,4113,132.41,428.06
private_office,471,701.88,812.26


#### 3.8.3. Scaling Error Identification
Flag potential decimal-placement errors when dividing an unusually high monthly value by 100 produces a plausible value for the same membership plan. Keep remaining unusual records flagged for review.

In [183]:
# Compare original monthly values with plan benchmarks
plan_median = df["plan_tier"].map(monthly_benchmarks["median"])
plan_p95 = df["plan_tier"].map(monthly_benchmarks["p95"])

monthly_original = df["lifetime_value"] / membership_months
monthly_corrected = monthly_original / 100

# Identify likely scaling errors
df["flag_ltv_scaling_error"] = (
    membership_months.ge(1) &
    (monthly_original > plan_p95 * 10) &
    monthly_corrected.between(plan_median * 0.25, plan_p95 * 2)
).fillna(False)

# Inspect candidates without changing the original values
scaling_candidates = df.loc[df["flag_ltv_scaling_error"], [
    "member_id", "plan_tier", "lifetime_value"
]].copy()

scaling_candidates["monthly_original"] = monthly_original
scaling_candidates["monthly_corrected"] = monthly_corrected

print(f"Potential scaling errors: {len(scaling_candidates):,}")
display(scaling_candidates.head(15))

# Check extreme values not explained by this rule
unresolved = (
    (monthly_original > plan_p95 * 5) &
    ~df["flag_ltv_scaling_error"]
)

print(f"Other unusually high monthly values: {unresolved.sum():,}")
display(df.loc[unresolved, ["member_id", "plan_tier", "lifetime_value"]].head(15))

Potential scaling errors: 78


,member_id,plan_tier,lifetime_value,monthly_original,monthly_corrected
320,CW000313,flex,47029.00,18836.35,188.36
374,CW000366,dedicated_desk,1342807.00,46396.19,463.96
973,CW000948,hot_desk,23879.00,11916.01,119.16
978,CW000953,flex,66238.00,59302.49,593.02
1059,CW001030,flex,193955.00,28384.57,283.85
1252,CW001218,hot_desk,29875.00,11965.72,119.66
1368,CW001332,flex,659184.00,30728.27,307.28
1379,CW001343,hot_desk,23066.00,11900.49,119.00
1402,CW001364,hot_desk,40828.00,24368.71,243.69
1532,CW001493,hot_desk,17653.00,12212.67,122.13


Other unusually high monthly values: 6


,member_id,plan_tier,lifetime_value
1494,CW001455,full_time,39661.00
3231,CW003134,full_time,5480.00
6246,CW006074,private_office,128431.81
8606,CW008360,full_time,11590.00
14058,CW013647,flex,22571.85
14118,CW013705,flex,17559.00


In [184]:
# Preserve original values and create a provisional corrected version
df["lifetime_value_original"] = df["lifetime_value"].copy()
df["lifetime_value_adjusted"] = df["lifetime_value"].copy()

# Apply the proposed scaling correction only to identified candidates
df.loc[df["flag_ltv_scaling_error"], "lifetime_value_adjusted"] /= 100

# Flag remaining unusual values, including records with missing plan information
review_mask = (
    unresolved.fillna(False) |
    (
        df["lifetime_value"].gt(100000) &
        (df["plan_tier"].isna() | membership_months.isna())
    )
)

df["flag_ltv_needs_review"] = (
    review_mask & ~df["flag_ltv_scaling_error"]
).fillna(False)

# Summarize the results
print(f"Provisionally corrected values: {df['flag_ltv_scaling_error'].sum():,}")
print(f"Remaining cases requiring review: {df['flag_ltv_needs_review'].sum():,}")
print(f"Original maximum: €{df['lifetime_value_original'].max():,.2f}")
print(f"Adjusted maximum: €{df['lifetime_value_adjusted'].max():,.2f}")

# Inspect unresolved cases
display(
    df.loc[df["flag_ltv_needs_review"], [
        "member_id", "plan_tier", "lifetime_value_original",
        "lifetime_value_adjusted"
    ]].sort_values("lifetime_value_original", ascending=False)
)

Provisionally corrected values: 78
Remaining cases requiring review: 8
Original maximum: €3,371,260.00
Adjusted maximum: €2,532,335.00


,member_id,plan_tier,lifetime_value_original,lifetime_value_adjusted
12903,CW012530,NaN,2532335.00,2532335.00
11151,CW010830,NaN,1154562.00,1154562.00
6246,CW006074,private_office,128431.81,128431.81
1494,CW001455,full_time,39661.00,39661.00
14058,CW013647,flex,22571.85,22571.85
14118,CW013705,flex,17559.00,17559.00
8606,CW008360,full_time,11590.00,11590.00
3231,CW003134,full_time,5480.00,5480.00


In [185]:
# Identify the two additional candidates with missing membership plans
missing_plan_candidates = (
    df["member_id"].isin(["CW012530", "CW010830"]) &
    df["flag_ltv_needs_review"]
)

# Apply the provisional /100 correction
df.loc[missing_plan_candidates, "lifetime_value_adjusted"] = (
    df.loc[missing_plan_candidates, "lifetime_value_original"] / 100
)

# Update the quality flags
df.loc[missing_plan_candidates, "flag_ltv_scaling_error"] = True
df.loc[missing_plan_candidates, "flag_ltv_needs_review"] = False

# Summarize the final provisional corrections
print(f"Scaling corrections: {df['flag_ltv_scaling_error'].sum():,}")
print(f"Remaining review cases: {df['flag_ltv_needs_review'].sum():,}")
print(f"Original maximum: €{df['lifetime_value_original'].max():,.2f}")
print(f"Adjusted maximum: €{df['lifetime_value_adjusted'].max():,.2f}")

display(
    df.loc[df["flag_ltv_needs_review"], [
        "member_id", "plan_tier",
        "lifetime_value_original", "lifetime_value_adjusted"
    ]]
)

Scaling corrections: 80
Remaining review cases: 6
Original maximum: €3,371,260.00
Adjusted maximum: €128,431.81


,member_id,plan_tier,lifetime_value_original,lifetime_value_adjusted
1494,CW001455,full_time,39661.00,39661.00
3231,CW003134,full_time,5480.00,5480.00
6246,CW006074,private_office,128431.81,128431.81
8606,CW008360,full_time,11590.00,11590.00
14058,CW013647,flex,22571.85,22571.85
14118,CW013705,flex,17559.00,17559.00


Some unusually high lifetime values may reflect amounts recorded in cents but interpreted as euros, or other decimal-placement errors. This is a plausible explanation rather than a confirmed source-data issue.

#### 3.8.4. Manual Review of Scaling Errors
Six additional records were reviewed individually using their membership duration and implied monthly revenue. The following correction factors are based on financial plausibility, not confirmed accounting records.
| Member ID | Factor | Rationale |
|---|---|---|
| CW001455 | ÷100 | Plausible full-time monthly revenue; membership slightly below one month |
| CW003134 | ÷100 | More plausible monthly value, but very short duration creates uncertainty |
| CW006074 | ÷10 | More plausible private-office monthly revenue than ÷100 |
| CW008360 | ÷100 | Plausible full-time monthly value; failed the duration threshold |
| CW013647 | ÷10 | Plausible Flex monthly revenue; ÷100 would be unusually low |
| CW013705 | ÷100 | Plausible Flex monthly value; membership slightly below one month |
**Decision:** Apply the reviewed factors, retain original amounts and correction flags, and assess their influence during clustering sensitivity analysis.

In [186]:
# Define scaling factors for the six manually reviewed records
manual_factors = {
    "CW001455": 100,
    "CW003134": 100,
    "CW006074": 10,
    "CW008360": 100,
    "CW013647": 10,
    "CW013705": 100
}

# Preserve the factors used for the 80 previously identified corrections
df["ltv_scaling_factor"] = 1
df.loc[df["flag_ltv_scaling_error"], "ltv_scaling_factor"] = 100

# Record manual corrections separately for traceability
df["flag_ltv_manual_correction"] = df["member_id"].isin(manual_factors)

manual_mask = df["flag_ltv_manual_correction"]

df.loc[manual_mask, "ltv_scaling_factor"] = (
    df.loc[manual_mask, "member_id"].map(manual_factors).astype(int)
)

# Update flags
df.loc[manual_mask, "flag_ltv_scaling_error"] = True
df.loc[manual_mask, "flag_ltv_needs_review"] = False

# Recalculate from original values to prevent repeated division
df["lifetime_value_adjusted"] = (
    df["lifetime_value_original"] / df["ltv_scaling_factor"]
)

# Summarize the corrections
print(f"Total scaling corrections: {df['flag_ltv_scaling_error'].sum():,}")
print(f"Divided by 100: {(df['ltv_scaling_factor'] == 100).sum():,}")
print(f"Divided by 10: {(df['ltv_scaling_factor'] == 10).sum():,}")
print(f"Manual corrections: {manual_mask.sum():,}")
print(f"Remaining review cases: {df['flag_ltv_needs_review'].sum():,}")
print(f"Adjusted maximum: €{df['lifetime_value_adjusted'].max():,.2f}")

# Verify the six manually corrected records
display(
    df.loc[manual_mask, [
        "member_id",
        "lifetime_value_original",
        "ltv_scaling_factor",
        "lifetime_value_adjusted"
    ]].sort_values("member_id")
)

Total scaling corrections: 86
Divided by 100: 84
Divided by 10: 2
Manual corrections: 6
Remaining review cases: 0
Adjusted maximum: €42,139.17


,member_id,lifetime_value_original,ltv_scaling_factor,lifetime_value_adjusted
1494,CW001455,39661.00,100,396.61
3231,CW003134,5480.00,100,54.80
6246,CW006074,128431.81,10,12843.18
8606,CW008360,11590.00,100,115.90
14058,CW013647,22571.85,10,2257.18
14118,CW013705,17559.00,100,175.59


#### 3.8.5. Negative Lifetime Values
The EDA notes that small negative lifetime values may reflect refunds or accounting adjustments. The data do not establish that all 82 negative records are sign-entry errors.
**Decision:** Preserve negative amounts as recorded rather than applying `abs()`. Flag them for financial feature selection and use a signed transformation, a separate financial treatment or a sensitivity check if required by clustering.

In [187]:
# Preserve the signed values after scaling corrections and flag negative financial amounts
# lifetime_value_adjusted was already calculated from lifetime_value_original / ltv_scaling_factor
assert np.allclose(
    df["lifetime_value_adjusted"].to_numpy(dtype=float),
    (df["lifetime_value_original"] / df["ltv_scaling_factor"]).to_numpy(dtype=float),
    equal_nan=True
)
df["flag_ltv_negative"] = df["lifetime_value_adjusted"].lt(0).fillna(False)
print(f"Negative LTV values preserved: {df['flag_ltv_negative'].sum():,}")
print(f"Scaling corrections applied: {df['flag_ltv_scaling_error'].sum():,}")
print(f"Missing adjusted LTV values: {df['lifetime_value_adjusted'].isna().sum():,}")

Negative LTV values preserved: 82
Scaling corrections applied: 86
Missing adjusted LTV values: 1,040


### 3.9. Behavioral Consistency Checks
The EDA identified inconsistencies between total visits, recent visits, weekly visit averages and entitled visits.
We investigate these contradictions before applying corrections, since some may reflect different observation periods rather than data-entry errors.

In [188]:
# Identify inconsistencies in visit-related variables
df["flag_zero_entitled_visits"] = df["entitled_visits_last_period"].eq(0).fillna(False)
df["flag_period_visits_exceed_total"] = df["actual_visits_last_period"].gt(df["total_visits"]).fillna(False)
df["flag_weekly_visits_without_total"] = (df["avg_weekly_visits"].gt(0) & df["total_visits"].eq(0)).fillna(False)
# Summarize the inconsistencies
visit_flags = [
    "flag_zero_entitled_visits",
    "flag_period_visits_exceed_total",
    "flag_weekly_visits_without_total"
]
visit_summary = pd.DataFrame({
    "Affected members": df[visit_flags].sum(),
    "Percentage (%)": (df[visit_flags].mean() * 100).round(2)
})
display(visit_summary)
# Inspect examples of each inconsistency
columns = [
    "member_id", "plan_tier", "membership_start",
    "total_visits", "avg_weekly_visits",
    "entitled_visits_last_period", "actual_visits_last_period"
]
for flag in visit_flags:
    print(f"\n{flag}")
    display(df.loc[df[flag], columns].head(10))

,Affected members,Percentage (%)
flag_zero_entitled_visits,10,0.06
flag_period_visits_exceed_total,1127,7.29
flag_weekly_visits_without_total,109,0.71



flag_zero_entitled_visits


,member_id,plan_tier,membership_start,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period
1988,CW001931,hot_desk,2023-06-03,7.00,0.66,0.00,6
2381,CW002313,full_time,2021-02-22,48.00,1.04,0.00,9
2382,CW002314,flex,2022-05-14,127.00,2.72,0.00,26
4695,CW004557,flex,2023-03-01,122.00,2.80,0.00,21
6973,CW006783,private_office,2021-05-13,134.00,1.00,0.00,9
7862,CW007640,flex,2021-08-25,186.00,2.63,0.00,23
10654,CW010353,hot_desk,2022-01-02,2.00,0.25,0.00,2
11708,CW011372,full_time,2021-10-03,1.00,0.01,0.00,0
12684,CW012315,hot_desk,2023-11-11,9.00,1.24,0.00,10
13866,CW013463,full_time,2022-07-03,1.00,0.00,0.00,0



flag_period_visits_exceed_total


,member_id,plan_tier,membership_start,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period
3,CW000004,flex,2022-01-02,8.00,1.23,18.74,9
16,CW000017,hot_desk,2022-06-03,1.00,0.33,8.78,2
34,CW000033,hot_desk,2022-12-23,7.00,0.87,16.40,9
40,CW000039,flex,2022-07-31,12.00,1.38,25.45,13
45,CW000044,hot_desk,2023-04-15,7.00,0.95,15.24,8
55,CW000054,flex,2023-11-08,20.00,2.61,23.06,22
75,CW000074,hot_desk,2023-11-23,8.00,1.39,11.11,10
80,CW000078,hot_desk,2023-08-15,5.00,0.81,12.02,6
84,CW000082,flex,2021-06-21,9.00,1.38,19.89,11
86,CW000084,hot_desk,2022-05-06,4.00,0.60,13.03,5



flag_weekly_visits_without_total


,member_id,plan_tier,membership_start,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period
181,CW000176,hot_desk,2021-02-01,0.00,1.88,17.39,17
239,CW000234,full_time,2021-11-25,0.00,0.06,43.48,1
255,CW000250,hot_desk,2022-10-09,0.00,1.17,17.39,10
377,CW000369,flex,2021-09-19,0.00,2.85,26.09,26
404,CW000394,flex,2021-05-20,0.00,2.23,26.09,13
426,CW015304,hot_desk,2022-07-13,0.00,0.68,17.39,7
705,CW000686,dedicated_desk,2022-04-26,0.00,0.09,43.48,1
751,CW000731,private_office,2021-08-22,0.00,5.27,60.87,53
985,CW000960,private_office,2021-01-16,0.00,0.72,60.87,7
1136,CW001105,hot_desk,2022-12-29,0.00,0.53,17.39,4


In [189]:
# Measure the difference between period and total visits
visit_gap = df["actual_visits_last_period"] - df["total_visits"]
inconsistent_gap = visit_gap.loc[df["flag_period_visits_exceed_total"]]
print("Visit discrepancies:")
display(inconsistent_gap.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("Excess Period Visits"))
print(f"Difference of 1–2 visits: {inconsistent_gap.between(1, 2).sum():,}")
print(f"Difference of 3–5 visits: {inconsistent_gap.between(3, 5).sum():,}")
print(f"Difference above 5 visits: {(inconsistent_gap > 5).sum():,}")
# Check overlaps between inconsistencies
print(f"\nZero-total cases also exceeding total: {(df['flag_weekly_visits_without_total'] & df['flag_period_visits_exceed_total']).sum():,}")
print(f"Zero-total cases with positive period visits: {(df['flag_weekly_visits_without_total'] & df['actual_visits_last_period'].gt(0)).sum():,}")
# Inspect the largest contradictions
review_columns = ["member_id", "plan_tier", "total_visits", "avg_weekly_visits", "actual_visits_last_period"]
largest_gaps = df.loc[df["flag_period_visits_exceed_total"], review_columns].copy()
largest_gaps["visit_difference"] = inconsistent_gap
display(largest_gaps.sort_values("visit_difference", ascending=False).head(15))

Visit discrepancies:


,Excess Period Visits
count,1127.00
mean,2.60
std,5.16
min,1.00
50%,1.00
75%,2.00
90%,4.00
95%,10.00
99%,28.96
max,53.00


Difference of 1–2 visits: 961
Difference of 3–5 visits: 79
Difference above 5 visits: 87

Zero-total cases also exceeding total: 109
Zero-total cases with positive period visits: 109


,member_id,plan_tier,total_visits,avg_weekly_visits,actual_visits_last_period,visit_difference
751,CW000731,private_office,0.00,5.27,53,53.00
2987,CW002896,dedicated_desk,0.00,3.66,41,41.00
12128,CW011779,private_office,0.00,5.12,40,40.00
6960,CW006770,full_time,0.00,4.25,39,39.00
9826,CW009555,dedicated_desk,0.00,4.47,35,35.00
12119,CW011770,full_time,0.00,3.98,35,35.00
2906,CW002817,dedicated_desk,0.00,4.00,34,34.00
14883,CW014453,full_time,0.00,3.74,33,33.00
5944,CW005780,private_office,0.00,5.01,31,31.00
14037,CW013627,dedicated_desk,0.00,3.54,31,31.00


**Visit Consistency Corrections**

The analysis identified 1,127 records where visits during the last period exceed total lifetime visits. Most discrepancies are small: 961 differ by only 1–2 visits, while 166 have larger differences.

**Decision:** 

Apply a minimal correction to the 961 small discrepancies by setting total visits equal to the recorded last-period visits. Mark the remaining 166 total-visit values as missing for later imputation.
All 109 cases with positive weekly visits but zero total visits are already included in these discrepancies. Additionally, 10 zero-entitlement values are treated as missing because they are inconsistent with the recorded membership activity.
Original values and quality flags are preserved for traceability.

In [190]:
# Preserve original visit and entitlement values
df["total_visits_original"] = df["total_visits"].copy()
df["entitled_visits_original"] = df["entitled_visits_last_period"].copy()
# Identify small and large inconsistencies
small_visit_gap = df["flag_period_visits_exceed_total"] & visit_gap.between(1, 2)
large_visit_gap = df["flag_period_visits_exceed_total"] & (visit_gap > 2)
# Preserve correction flags
df["flag_total_visits_minimum_fix"] = small_visit_gap.fillna(False)
df["flag_total_visits_needs_imputation"] = large_visit_gap.fillna(False)
# Apply minimum-value correction for small discrepancies
df.loc[small_visit_gap, "total_visits"] = df.loc[small_visit_gap, "actual_visits_last_period"]
# Mark larger discrepancies as missing
df.loc[large_visit_gap, "total_visits"] = np.nan
# Mark invalid zero entitlements as missing
df.loc[df["flag_zero_entitled_visits"], "entitled_visits_last_period"] = np.nan
# Verify the corrections
print(f"Minimum-value corrections: {small_visit_gap.sum():,}")
print(f"Total visits marked missing: {large_visit_gap.sum():,}")
print(f"Zero entitlements marked missing: {df['flag_zero_entitled_visits'].sum():,}")
print(f"Remaining visit contradictions: {(df['total_visits'] < df['actual_visits_last_period']).sum():,}")
print(f"Total missing visits: {df['total_visits'].isna().sum():,}")
print(f"Missing entitlements: {df['entitled_visits_last_period'].isna().sum():,}")

Minimum-value corrections: 961
Total visits marked missing: 166
Zero entitlements marked missing: 10
Remaining visit contradictions: 0
Total missing visits: 208
Missing entitlements: 10


### 3.10. Zero Lifetime Value Treatment
The EDA identified an association between zero LTV and the original recency placeholder `999`. Previous activity checks also found evidence of membership use among the remaining zero-LTV records.
**Decision:** Reclassify zero LTV as potentially missing financial information, preserving separate flags for records with and without the original `999` placeholder. Genuine zero-revenue cases remain possible; assess the effect of this assumption on Value clustering.

In [191]:
# Recover original recency information from the raw dataset
raw_recency = pd.to_numeric(df_raw.drop_duplicates("member_id").set_index("member_id")["days_since_last_visit"], errors="coerce")
df["flag_original_recency_999"] = df["member_id"].map(raw_recency).eq(999).fillna(False)
# Identify zero-LTV records and preserve their two missingness origins
df["flag_ltv_zero"] = df["lifetime_value_adjusted"].eq(0).fillna(False)
df["flag_ltv_structural_missing"] = df["flag_ltv_zero"] & df["flag_original_recency_999"]
df["flag_ltv_additional_zero_missing"] = df["flag_ltv_zero"] & ~df["flag_original_recency_999"]
df["flag_ltv_zero_reclassified"] = df["flag_ltv_zero"].copy()
# Reclassify zeros as missing for subsequent financial imputation
df.loc[df["flag_ltv_zero_reclassified"], "lifetime_value_adjusted"] = np.nan
print(f"Zeros with recency 999: {df['flag_ltv_structural_missing'].sum():,}")
print(f"Other zeros: {df['flag_ltv_additional_zero_missing'].sum():,}")
print(f"Total zeros reclassified: {df['flag_ltv_zero_reclassified'].sum():,}")
print(f"Remaining zeros: {df['lifetime_value_adjusted'].eq(0).sum():,}")
print(f"Missing adjusted LTV: {df['lifetime_value_adjusted'].isna().sum():,}")

Zeros with recency 999: 1,296
Other zeros: 446
Total zeros reclassified: 1,742
Remaining zeros: 0
Missing adjusted LTV: 2,782


### 3.11. Temporal Consistency Review
Review remaining chronological inconsistencies involving membership start dates, end dates and badge swipes.
Preserve valid membership information wherever possible and avoid correcting dates without sufficient evidence, as these variables influence tenure and recency features.

In [192]:
# Reassess chronological inconsistencies
days_after_end = (df["last_badge_swipe"] - df["membership_end"]).dt.days
temporal_checks = pd.DataFrame({
    "End before start": df["membership_end"].lt(df["membership_start"]).fillna(False),
    "Swipe before start": df["last_badge_swipe"].lt(df["membership_start"]).fillna(False),
    "Swipe over 30 days after end": days_after_end.gt(30).fillna(False),
    "Swipe after reference date": df["last_badge_swipe"].gt(reference_date).fillna(False)
}, index=df.index)
# Summarize affected records
temporal_summary = pd.DataFrame({
    "Affected members": temporal_checks.sum(),
    "Percentage (%)": (temporal_checks.mean() * 100).round(2)
})
display(temporal_summary)
print(f"Members with at least one temporal issue: {temporal_checks.any(axis=1).sum():,}")
# Inspect records with membership end before start
columns = ["member_id", "membership_start", "membership_end", "last_badge_swipe", "is_active"]
display(df.loc[temporal_checks["End before start"], columns].head(10))
# Inspect badge swipes before membership starts
display(df.loc[temporal_checks["Swipe before start"], columns].head(10))

,Affected members,Percentage (%)
End before start,152,0.98
Swipe before start,94,0.61
Swipe over 30 days after end,174,1.13
Swipe after reference date,34,0.22


Members with at least one temporal issue: 310


,member_id,membership_start,membership_end,last_badge_swipe,is_active
165,CW000161,2021-10-25,2021-10-17,2023-02-12,0
290,CW000284,2021-03-19,2021-03-12,2022-08-11,0
322,CW000315,2021-08-17,2021-06-29,2021-08-17,0
493,CW000480,2021-10-09,2021-09-15,2022-05-13,0
511,CW000498,2022-05-10,2022-02-10,2023-11-15,0
601,CW000585,2022-09-20,2022-08-19,2023-07-17,0
1047,CW001018,2022-03-09,2021-12-17,2023-12-21,0
1118,CW001087,2021-06-26,2021-04-03,2023-03-18,0
1157,CW001125,2023-12-06,2023-10-01,2023-12-27,0
1295,CW001259,2022-03-10,2022-02-22,2023-09-02,0


,member_id,membership_start,membership_end,last_badge_swipe,is_active
466,CW000454,2021-09-16,NaT,2021-07-14,1
556,CW015291,2022-01-06,2022-04-07,2021-12-04,0
679,CW000660,2021-01-02,2023-12-07,2020-12-12,0
735,CW000715,2023-12-26,NaT,2023-10-30,1
778,CW000756,2021-11-19,2023-03-23,2021-11-12,0
1045,CW001016,2023-12-24,NaT,2023-08-31,1
1090,CW001061,2021-12-02,2022-01-19,2021-10-17,0
1306,CW001270,2022-06-09,2023-10-05,2022-04-15,0
1370,CW001334,2021-01-18,2023-05-29,2020-11-11,0
1414,CW001376,2021-10-06,2022-09-26,2021-07-20,0


**Potential Date Corrections**

For records with inconsistent dates, we evaluate whether swapping the affected dates produces a valid membership timeline and financially plausible monthly lifetime value.
A correction is considered only when the resulting chronology is consistent and the implied monthly revenue is reasonable for the membership plan. These checks identify candidates for correction rather than proving which original date is incorrect.

In [193]:
# Evaluate alternative date arrangements without changing the dataset
date_scenarios = []
for scenario, mask in {
    "Swap start/end": temporal_checks["End before start"],
    "Swap start/swipe": temporal_checks["Swipe before start"]
}.items():
    check = df.loc[mask, ["member_id", "plan_tier", "membership_start", "membership_end", "last_badge_swipe", "lifetime_value_adjusted"]].copy()
    if scenario == "Swap start/end":
        new_start = check["membership_end"]
        new_end = check["membership_start"]
        new_swipe = check["last_badge_swipe"]
    else:
        new_start = check["last_badge_swipe"]
        new_end = check["membership_end"]
        new_swipe = check["membership_start"]
    duration_days = (new_end.fillna(reference_date) - new_start).dt.days
    check["scenario"] = scenario
    check["new_start"] = new_start
    check["new_end"] = new_end
    check["new_swipe"] = new_swipe
    check["new_months"] = duration_days / 30.44
    check["new_monthly_ltv"] = (check["lifetime_value_adjusted"] / check["new_months"]).where(duration_days >= 30)
    check["plan_median"] = check["plan_tier"].map(monthly_benchmarks["median"])
    check["plan_p95"] = check["plan_tier"].map(monthly_benchmarks["p95"])
    check["valid_chronology"] = (
        duration_days.ge(30) &
        new_swipe.ge(new_start) &
        (new_end.isna() | new_swipe.le(new_end + pd.Timedelta(days=30))) &
        new_swipe.le(reference_date)
    ).fillna(False)
    check["plausible_ltv"] = check["new_monthly_ltv"].between(
        check["plan_median"] * 0.25,
        check["plan_p95"] * 2
    ).fillna(False)
    check["candidate_correction"] = check["valid_chronology"] & check["plausible_ltv"]
    date_scenarios.append(check)
date_review = pd.concat(date_scenarios)
display(date_review.groupby("scenario")[["valid_chronology", "plausible_ltv", "candidate_correction"]].sum())
print(f"Total candidate corrections: {date_review['candidate_correction'].sum():,}")
display(date_review.loc[date_review["candidate_correction"], ["member_id", "scenario", "new_start", "new_end", "new_swipe", "new_months", "new_monthly_ltv", "plan_median"]].head(20))

C:\Users\Mister\AppData\Local\Temp\ipykernel_9692\4057782884.py:28: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  (new_end.isna() | new_swipe.le(new_end + pd.Timedelta(days=30))) &
C:\Users\Mister\AppData\Local\Temp\ipykernel_9692\4057782884.py:28: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  (new_end.isna() | new_swipe.le(new_end + pd.Timedelta(days=30))) &


,valid_chronology,plausible_ltv,candidate_correction
scenario,,,
Swap start/end,20,34,13
Swap start/swipe,92,70,70


Total candidate corrections: 83


,member_id,scenario,new_start,new_end,new_swipe,new_months,new_monthly_ltv,plan_median
1157,CW001125,Swap start/end,2023-10-01,2023-12-06,2023-12-27,2.17,90.26,204.83
3466,CW003359,Swap start/end,2022-08-14,2022-10-09,2022-11-04,1.84,191.28,204.83
4644,CW004508,Swap start/end,2022-05-29,2022-08-15,2022-08-15,2.56,100.36,132.41
5098,CW015408,Swap start/end,2023-10-24,2023-12-17,2023-12-31,1.77,43.15,132.41
5136,CW004989,Swap start/end,2023-10-06,2023-12-13,2023-12-13,2.23,40.73,132.41
5212,CW005061,Swap start/end,2022-07-11,2022-10-02,2022-10-02,2.73,96.04,204.83
6559,CW006381,Swap start/end,2022-05-01,2022-06-02,2022-06-02,1.05,166.09,132.41
9709,CW009438,Swap start/end,2021-09-21,2021-11-21,2021-11-25,2.00,193.80,312.67
10260,CW009973,Swap start/end,2021-04-21,2021-06-04,2021-06-18,1.45,122.87,132.41
10717,CW010416,Swap start/end,2023-10-24,2023-12-17,2023-12-31,1.77,43.15,132.41


In [194]:
# Check whether proposed date swaps preserve recorded recency
candidates = date_review.loc[date_review["candidate_correction"]].copy()
ref_date = pd.Timestamp(reference_date)
# Calculate recency using the current dates
current_recency = (df["membership_end"].fillna(ref_date) - df["last_badge_swipe"]).dt.days
# Calculate recency using the proposed dates
proposed_end = pd.to_datetime(candidates["new_end"], errors="coerce")
proposed_swipe = pd.to_datetime(candidates["new_swipe"], errors="coerce")
proposed_recency = (proposed_end.fillna(ref_date) - proposed_swipe).dt.days
# Compare both with the recorded recency
observed_recency = pd.to_numeric(df["days_since_last_visit"], errors="coerce").reindex(candidates.index)
candidates["observed_recency"] = observed_recency.to_numpy()
candidates["current_error_days"] = observed_recency.to_numpy() - current_recency.reindex(candidates.index).to_numpy()
candidates["proposed_error_days"] = observed_recency.to_numpy() - proposed_recency.to_numpy()
candidates["current_matches"] = candidates["current_error_days"].abs().le(1)
candidates["proposed_matches"] = candidates["proposed_error_days"].abs().le(1)
# Summarize the validation
print(f"Proposed corrections: {len(candidates):,}")
print(f"Unique members: {candidates['member_id'].nunique():,}")
display(candidates.groupby("scenario").agg(
    candidates=("member_id", "size"),
    available_recency=("observed_recency", "count"),
    current_matches=("current_matches", "sum"),
    proposed_matches=("proposed_matches", "sum")
))
# Identify swaps that would break an existing recency match
worse_swaps = candidates["current_matches"] & ~candidates["proposed_matches"]
print(f"Corrections that would break recency consistency: {worse_swaps.sum():,}")
display(candidates.loc[worse_swaps, ["member_id", "scenario", "current_error_days", "proposed_error_days"]].head(15))

Proposed corrections: 83
Unique members: 83


,candidates,available_recency,current_matches,proposed_matches
scenario,,,,
Swap start/end,13,11,0,0
Swap start/swipe,70,70,26,8


Corrections that would break recency consistency: 26


,member_id,scenario,current_error_days,proposed_error_days
778,CW000756,Swap start/swipe,0.00,7.00
1567,CW001527,Swap start/swipe,0.00,77.00
2453,CW002382,Swap start/swipe,0.00,107.00
3126,CW003031,Swap start/swipe,0.00,85.00
4270,CW004145,Swap start/swipe,0.00,63.00
4459,CW004328,Swap start/swipe,0.00,117.00
5569,CW005412,Swap start/swipe,0.00,36.00
5760,CW005597,Swap start/swipe,0.00,93.00
5806,CW005643,Swap start/swipe,0.00,75.00
7101,CW006905,Swap start/swipe,0.00,103.00


**Date Swap Validation**

Of the 84 potential corrections, 26 start/swipe swaps would break an existing recency match, while 8 would establish a new match. The remaining candidates lack sufficient recency evidence to establish the correct date arrangement.

**Decision:**

 Provisionally apply the 8 start/swipe swaps supported by chronology, financial plausibility and recency consistency. Preserve the original dates and flag corrected records. Leave other inconsistencies unresolved rather than introducing potentially incorrect dates.

In [195]:
# Preserve original dates before applying corrections
for col in ["membership_start", "membership_end", "last_badge_swipe"]:
    original_col = f"{col}_original"
    if original_col not in df.columns:
        df[original_col] = df[col].copy()
# Select swaps that improve recency consistency
valid_swaps = candidates.loc[
    (candidates["scenario"] == "Swap start/swipe") &
    candidates["proposed_matches"].fillna(False) &
    ~candidates["current_matches"].fillna(False)
]
# Apply the validated date arrangements
swap_idx = valid_swaps.index
df.loc[swap_idx, "membership_start"] = valid_swaps["new_start"]
df.loc[swap_idx, "last_badge_swipe"] = valid_swaps["new_swipe"]
# Record corrections
df["flag_date_swap_start_swipe"] = df.index.isin(swap_idx)
# Verify results
print(f"Date swaps applied: {len(swap_idx):,}")
print(f"Remaining end-before-start cases: {df['membership_end'].lt(df['membership_start']).sum():,}")
print(f"Remaining swipe-before-start cases: {df['last_badge_swipe'].lt(df['membership_start']).sum():,}")
display(df.loc[swap_idx, ["member_id", "membership_start_original", "last_badge_swipe_original", "membership_start", "last_badge_swipe"]])

Date swaps applied: 8
Remaining end-before-start cases: 152
Remaining swipe-before-start cases: 86


,member_id,membership_start_original,last_badge_swipe_original,membership_start,last_badge_swipe
5431,CW005277,2022-11-20,2022-11-09,2022-11-09,2022-11-20
6748,CW006564,2021-06-19,2021-02-28,2021-02-28,2021-06-19
7815,CW015306,2022-07-08,2022-07-04,2022-07-04,2022-07-08
8765,CW008515,2021-08-18,2021-04-25,2021-04-25,2021-08-18
10733,CW015449,2022-11-08,2022-11-04,2022-11-04,2022-11-08
14072,CW013659,2021-07-25,2021-05-07,2021-05-07,2021-07-25
14509,CW015434,2023-11-15,2023-11-11,2023-11-11,2023-11-15
15254,CW014809,2023-07-27,2023-04-06,2023-04-06,2023-07-27


**Final Temporal Cleaning Decisions**

Eight start/swipe date swaps were applied after validation against chronology, monthly lifetime value and recorded recency.
The remaining inconsistencies were not corrected because the available evidence does not reliably identify which date is incorrect. Original dates are preserved, and quality flags are maintained.
Invalid membership durations will be excluded from tenure calculations, while questionable badge-swipe dates will be flagged for recency-related feature engineering. These flags support data cleaning and are not automatically used as clustering features.

In [196]:
# Recalculate temporal quality flags after the eight corrections
df["flag_date_end_before_start"] = df["membership_end"].lt(df["membership_start"]).fillna(False)
df["flag_date_swipe_before_start"] = df["last_badge_swipe"].lt(df["membership_start"]).fillna(False)
df["flag_date_swipe_after_end"] = ((df["last_badge_swipe"] - df["membership_end"]).dt.days > 30).fillna(False)
df["flag_date_swipe_after_reference"] = df["last_badge_swipe"].gt(reference_date).fillna(False)
# Identify dates that are unreliable for recency calculations
df["flag_unreliable_swipe"] = (
    df["flag_date_swipe_before_start"] |
    df["flag_date_swipe_after_end"] |
    df["flag_date_swipe_after_reference"]
)
# Recalculate tenure using the corrected dates
effective_end = df["membership_end"].fillna(reference_date)
tenure_days = (effective_end - df["membership_start"]).dt.days
df["membership_months_clean"] = (tenure_days / 30.44).where(tenure_days >= 0)
# Summarize remaining temporal issues
temporal_flags = [
    "flag_date_end_before_start",
    "flag_date_swipe_before_start",
    "flag_date_swipe_after_end",
    "flag_date_swipe_after_reference",
    "flag_unreliable_swipe"
]
display(df[temporal_flags].sum().to_frame("Affected members"))
print(f"Invalid or missing membership durations: {df['membership_months_clean'].isna().sum():,}")
print(f"Valid membership durations: {df['membership_months_clean'].notna().sum():,}")

,Affected members
flag_date_end_before_start,152
flag_date_swipe_before_start,86
flag_date_swipe_after_end,174
flag_date_swipe_after_reference,34
flag_unreliable_swipe,287


Invalid or missing membership durations: 152
Valid membership durations: 15,298


### 3.12. Missing Value Assessment
Following data cleaning, we reassess missing values across the variables relevant to customer segmentation.
This establishes the final imputation requirements and distinguishes genuinely missing information from intentionally unavailable dates or unreliable observations.

In [197]:
# Select variables requiring a missing-value assessment
imputation_columns = [
    "age", "gender", "country", "plan_tier",
    "n_colleagues_on_account", "plan_days_per_week",
    "total_visits", "n_renewals", "meeting_room_hours",
    "event_space_hours", "guest_passes_issued",
    "days_since_last_visit", "lifetime_value_adjusted",
    "entitled_visits_last_period", "membership_months_clean"
]
# Summarize missing values after cleaning
missing_summary = pd.DataFrame({
    "Missing": df[imputation_columns].isna().sum(),
    "Missing (%)": (df[imputation_columns].isna().mean() * 100).round(2),
    "Data Type": df[imputation_columns].dtypes.astype(str)
})
missing_summary = missing_summary.loc[missing_summary["Missing"] > 0]
display(missing_summary.sort_values("Missing (%)", ascending=False))
print(f"Members with at least one missing value: {df[imputation_columns].isna().any(axis=1).sum():,}")

,Missing,Missing (%),Data Type
lifetime_value_adjusted,2782,18.01,Float64
days_since_last_visit,1354,8.76,float64
age,700,4.53,float64
n_colleagues_on_account,565,3.66,float64
gender,530,3.43,object
country,429,2.78,object
meeting_room_hours,409,2.65,float64
guest_passes_issued,210,1.36,float64
total_visits,208,1.35,float64
membership_months_clean,152,0.98,float64


Members with at least one missing value: 5,102


### 3.13. Missing Value Treatment
Missing values will be handled using a combination of reliable reconstruction, simple imputation and predictive methods.
Where possible, values will be recovered from existing information before applying statistical imputation. Alternative methods will be compared using artificially masked observations and appropriate error metrics.

In [198]:
# Reconstruct recency using the same reference logic established in the EDA
effective_end = df["membership_end"].fillna(pd.Timestamp(reference_date))
recency_from_dates = (effective_end - df["last_badge_swipe"]).dt.days
# Restrict reconstruction to chronologically reliable records
reliable_dates = (
    ~df["flag_unreliable_swipe"] &
    ~df["flag_date_end_before_start"] &
    recency_from_dates.ge(0)
).fillna(False)
# Validate against existing recorded recency
observed_recency = df["days_since_last_visit"]
validation_mask = reliable_dates & observed_recency.notna()
recency_error = (observed_recency[validation_mask] - recency_from_dates[validation_mask]).abs()
# Identify missing values potentially recoverable from dates
recoverable_recency = reliable_dates & observed_recency.isna()
print(f"Existing recency values available for validation: {validation_mask.sum():,}")
print(f"Exact matches: {(recency_error == 0).sum():,}")
print(f"Matches within 1 day: {(recency_error <= 1).sum():,}")
print(f"Agreement within 1 day: {(recency_error <= 1).mean() * 100:.2f}%")
print(f"Missing recency values: {observed_recency.isna().sum():,}")
print(f"Potentially recoverable from dates: {recoverable_recency.sum():,}")
print(f"Remaining for imputation: {(observed_recency.isna() & ~recoverable_recency).sum():,}")
display(recency_error.describe(percentiles=[0.50, 0.90, 0.95, 0.99]).to_frame("Absolute Error (Days)"))

Existing recency values available for validation: 13,858
Exact matches: 13,120
Matches within 1 day: 13,791
Agreement within 1 day: 99.52%
Missing recency values: 1,354
Potentially recoverable from dates: 1,267
Remaining for imputation: 87


,Absolute Error (Days)
count,13858.00
mean,0.40
std,17.76
min,0.00
50%,0.00
90%,0.00
95%,1.00
99%,1.00
max,993.00


**Recency Reconstruction**

Date-based recency was validated against 13,858 existing observations, achieving 99.52% agreement within one day.
Of the 1,354 missing recency values, 1,267 can be reconstructed from chronologically reliable membership and badge-swipe dates.

**Decision:** 

Recover these values directly rather than applying statistical imputation. Preserve the original recency values and flag reconstructed records. The remaining 87 cases will be assessed separately because their dates do not satisfy the reliability criteria.

In [199]:
# Preserve the original recency values
if "days_since_last_visit_original" not in df.columns:
    df["days_since_last_visit_original"] = df["days_since_last_visit"].copy()
# Flag values reconstructed from reliable dates
df["flag_recency_reconstructed"] = recoverable_recency.fillna(False)
# Apply date-based reconstruction
df.loc[recoverable_recency, "days_since_last_visit"] = recency_from_dates.loc[recoverable_recency].astype(float)
# Verify reconstruction
print(f"Reconstructed recency values: {df['flag_recency_reconstructed'].sum():,}")
print(f"Remaining missing recency values: {df['days_since_last_visit'].isna().sum():,}")
print(f"Negative recency values: {df['days_since_last_visit'].lt(0).sum():,}")
# Inspect reconstructed values by original placeholder status
display(df.loc[df["flag_recency_reconstructed"]].groupby("flag_original_recency_999")["days_since_last_visit"].describe())

Reconstructed recency values: 1,267
Remaining missing recency values: 87
Negative recency values: 0


,count,mean,std,min,25%,50%,75%,max
flag_original_recency_999,,,,,,,,
True,1267.00,55.00,71.71,0.00,8.00,30.00,62.00,378.00


In [200]:
# Investigate the remaining missing recency values
remaining = df["days_since_last_visit"].isna()
original_recency = df["member_id"].map(raw_recency)
print(f"Remaining missing recency: {remaining.sum():,}")
print(f"Originally coded as 999: {(remaining & original_recency.eq(999).fillna(False)).sum():,}")
print(f"Originally negative: {(remaining & original_recency.lt(0).fillna(False)).sum():,}")
print(f"Unreliable badge swipes: {(remaining & df['flag_unreliable_swipe']).sum():,}")
print(f"Invalid membership durations: {(remaining & df['flag_date_end_before_start']).sum():,}")
# Inspect examples and their reliability flags
columns = [
    "member_id", "membership_start", "membership_end",
    "last_badge_swipe", "flag_unreliable_swipe",
    "flag_date_end_before_start", "flag_original_recency_999"
]
display(df.loc[remaining, columns].head(15))

Remaining missing recency: 87
Originally coded as 999: 31
Originally negative: 56
Unreliable badge swipes: 67
Invalid membership durations: 50


,member_id,membership_start,membership_end,last_badge_swipe,flag_unreliable_swipe,flag_date_end_before_start,flag_original_recency_999
0,CW000001,2021-08-01,2023-03-06,2023-03-23,False,False,False
322,CW000315,2021-08-17,2021-06-29,2021-08-17,True,True,True
493,CW000480,2021-10-09,2021-09-15,2022-05-13,True,True,False
511,CW000498,2022-05-10,2022-02-10,2023-11-15,True,True,False
557,CW000541,2022-01-06,2022-08-30,2022-12-10,True,False,False
811,CW000788,2021-12-20,NaT,2024-04-11,True,False,False
994,CW000969,2022-05-06,2022-06-26,2022-06-30,False,False,True
1047,CW001018,2022-03-09,2021-12-17,2023-12-21,True,True,False
1090,CW001061,2021-12-02,2022-01-19,2021-10-17,True,False,True
1118,CW001087,2021-06-26,2021-04-03,2023-03-18,True,True,True


In [201]:
# Investigate unresolved recency without an unreliable-swipe flag
remaining = df["days_since_last_visit"].isna()
not_flagged = remaining & ~df["flag_unreliable_swipe"]
recency_review = df.loc[not_flagged, [
    "member_id", "membership_start", "membership_end",
    "last_badge_swipe", "flag_date_end_before_start"
]].copy()
recency_review["calculated_recency"] = recency_from_dates.loc[not_flagged]
recency_review["days_after_end"] = (
    recency_review["last_badge_swipe"] - recency_review["membership_end"]
).dt.days
print(f"Unresolved without unreliable-swipe flag: {len(recency_review):,}")
print(f"Negative calculated recency: {recency_review['calculated_recency'].lt(0).sum():,}")
print(f"Invalid membership chronology: {recency_review['flag_date_end_before_start'].sum():,}")
display(recency_review)

Unresolved without unreliable-swipe flag: 20
Negative calculated recency: 20
Invalid membership chronology: 10


,member_id,membership_start,membership_end,last_badge_swipe,flag_date_end_before_start,calculated_recency,days_after_end
0,CW000001,2021-08-01,2023-03-06,2023-03-23,False,-17,17
994,CW000969,2022-05-06,2022-06-26,2022-06-30,False,-4,4
3480,CW003372,2021-04-04,2021-06-23,2021-07-21,False,-28,28
3756,CW003645,2022-02-17,2022-01-20,2022-02-17,True,-28,28
4463,CW004332,2022-04-25,2023-09-16,2023-10-08,False,-22,22
4654,CW004517,2021-07-11,2021-09-01,2021-09-18,False,-17,17
5573,CW005416,2022-04-08,2022-03-10,2022-04-08,True,-29,29
6301,CW006128,2021-01-26,2022-11-17,2022-12-03,False,-16,16
6996,CW006806,2021-09-11,2021-08-25,2021-09-14,True,-20,20
8890,CW008640,2021-12-02,2021-11-22,2021-12-02,True,-10,10


**Unresolved Recency Values**

Of the 1,354 missing recency values, 1,267 were reconstructed directly from reliable dates, leaving 87 unresolved cases.
Among these, 67 have unreliable badge-swipe dates. The remaining 20 produce negative calculated recency because their last recorded visit occurred after membership end, within the provisional 30-day tolerance.

**Decision:**

 Preserve these 87 values as missing for statistical imputation rather than introducing unsupported date corrections.

**Lifetime Value Imputation Comparison**

Because lifetime value has 18.01% missing values, different imputation methods will be evaluated before selecting a final approach.
A reproducible 80/20 split of observed LTV values will be used to compare median-based and predictive methods using MAE and RMSE.
The validation subset will remain unchanged across methods to ensure fair comparison. Since actual missingness may be systematic, reconstruction accuracy on observed values will be considered alongside the impact on customer segmentation.

In [202]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Select records with known lifetime values
ltv_known = df.loc[df["lifetime_value_adjusted"].notna()].copy()
# Split observed values for imputation evaluation
ltv_train_idx, ltv_valid_idx = train_test_split(ltv_known.index, test_size=0.20, random_state=42)
ltv_train = df.loc[ltv_train_idx]
ltv_valid = df.loc[ltv_valid_idx]
# Calculate medians using training data only
global_median = float(ltv_train["lifetime_value_adjusted"].median())
plan_medians = ltv_train.groupby("plan_tier")["lifetime_value_adjusted"].median()
# Generate predictions
y_true = ltv_valid["lifetime_value_adjusted"].to_numpy(dtype=float)
pred_global = np.full(len(ltv_valid), global_median)
pred_plan = ltv_valid["plan_tier"].map(plan_medians).fillna(global_median).to_numpy(dtype=float)
# Evaluate both imputation methods
ltv_results = pd.DataFrame([
    {"Method": "Global median", "MAE": mean_absolute_error(y_true, pred_global), "RMSE": np.sqrt(mean_squared_error(y_true, pred_global))},
    {"Method": "Median by plan", "MAE": mean_absolute_error(y_true, pred_plan), "RMSE": np.sqrt(mean_squared_error(y_true, pred_plan))}
])
print(f"Training observations: {len(ltv_train):,}")
print(f"Validation observations: {len(ltv_valid):,}")
display(ltv_results.round(2))

Training observations: 10,134
Validation observations: 2,534


,Method,MAE,RMSE
0,Global median,3710.89,6209.15
1,Median by plan,2221.04,3703.04


In [203]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
# Select predictors without using LTV-derived variables
numeric_features = [
    "age", "n_colleagues_on_account", "plan_days_per_week",
    "membership_months_clean", "total_visits", "avg_weekly_visits",
    "actual_visits_last_period", "entitled_visits_last_period",
    "n_renewals", "meeting_room_hours", "event_space_hours",
    "guest_passes_issued", "days_since_last_visit"
]
categorical_features = ["plan_tier"]
model_features = numeric_features + categorical_features
# Prepare training and validation features
X_train = ltv_train[model_features]
X_valid = ltv_valid[model_features]
# Handle missing predictors and encode membership plans
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])
# Build and train the Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_true if False else ltv_train["lifetime_value_adjusted"].to_numpy(dtype=float))
# Predict the same validation observations
pred_rf = rf_pipeline.predict(X_valid)
# Add results to our existing comparison
rf_result = pd.DataFrame([{
    "Method": "Random Forest",
    "MAE": mean_absolute_error(y_true, pred_rf),
    "RMSE": np.sqrt(np.mean((y_true - pred_rf) ** 2))
}])
ltv_results = pd.concat([ltv_results, rf_result], ignore_index=True)
display(ltv_results.sort_values("MAE").round(2))

,Method,MAE,RMSE
2,Random Forest,366.15,1203.15
1,Median by plan,2221.04,3703.04
0,Global median,3710.89,6209.15


In [204]:
# Evaluate Random Forest errors across membership plans
rf_validation = ltv_valid[["plan_tier"]].copy()
rf_validation["actual_ltv"] = y_true
rf_validation["predicted_ltv"] = pred_rf
rf_validation["absolute_error"] = np.abs(y_true - pred_rf)
plan_performance = rf_validation.groupby("plan_tier", dropna=False).agg(
    members=("actual_ltv", "size"),
    mae=("absolute_error", "mean"),
    actual_median=("actual_ltv", "median"),
    predicted_median=("predicted_ltv", "median")
)
display(plan_performance.round(2))
# Compare customers with observed and missing lifetime values
df["flag_ltv_missing"] = df["lifetime_value_adjusted"].isna()
comparison_features = [
    "membership_months_clean", "total_visits",
    "avg_weekly_visits", "actual_visits_last_period",
    "n_renewals", "age"
]
display(df.groupby("flag_ltv_missing")[comparison_features].median().T.round(2))
# Compare membership-plan distributions
display(pd.crosstab(
    df["flag_ltv_missing"],
    df["plan_tier"],
    normalize="index"
).round(3))

,members,mae,actual_median,predicted_median
plan_tier,,,,
dedicated_desk,240,650.87,11241.82,10707.54
flex,924,243.86,1713.07,1748.90
full_time,401,599.87,5304.43,5295.93
hot_desk,854,169.16,344.10,342.67
private_office,108,1439.88,22013.80,21985.73
NaN,7,824.96,16668.81,17234.15


flag_ltv_missing,False,True
membership_months_clean,8.54,12.65
total_visits,34.00,43.50
avg_weekly_visits,1.45,1.28
actual_visits_last_period,11.00,10.00
n_renewals,0.00,1.00
age,34.00,34.00


plan_tier,dedicated_desk,flex,full_time,hot_desk,private_office
flag_ltv_missing,,,,,
False,0.10,0.35,0.16,0.35,0.04
True,0.12,0.35,0.19,0.28,0.06


**Random Forest Lifetime Value Imputation**
Compare the updated validation metrics after retaining negative LTV amounts. Random Forest is the provisional method selected during earlier evaluation, but its performance must be confirmed using the revised target values before estimating the missing LTV records. The cleaned values and imputation flags remain separate from the originals.

In [205]:
# Identify observed and missing lifetime values
ltv_observed_mask = df["lifetime_value_adjusted"].notna()
ltv_missing_mask = ~ltv_observed_mask
# Prevent applying a previously selected model if the updated comparison favors another method
best_ltv_method = ltv_results.sort_values("MAE").iloc[0]["Method"]
if best_ltv_method != "Random Forest":
    raise ValueError(f"Updated validation favors {best_ltv_method}; review LTV imputation before proceeding.")
# Refit on all observed values, including preserved legitimate negative amounts
rf_pipeline.fit(df.loc[ltv_observed_mask, model_features], df.loc[ltv_observed_mask, "lifetime_value_adjusted"].to_numpy(dtype=float))
df["lifetime_value_imputed"] = df["lifetime_value_adjusted"].copy()
df["flag_ltv_imputed"] = ltv_missing_mask.copy()
df.loc[ltv_missing_mask, "lifetime_value_imputed"] = rf_pipeline.predict(df.loc[ltv_missing_mask, model_features])
print(f"Imputed LTV values: {df['flag_ltv_imputed'].sum():,}")
print(f"Remaining missing LTV: {df['lifetime_value_imputed'].isna().sum():,}")
print(f"Negative *predictions*: {df.loc[ltv_missing_mask, 'lifetime_value_imputed'].lt(0).sum():,}")
print(f"Recorded negative LTV retained: {df['flag_ltv_negative'].sum():,}")
ltv_imputation_summary = pd.DataFrame({
    "Observed LTV": df.loc[ltv_observed_mask, "lifetime_value_adjusted"].describe(),
    "Imputed LTV": df.loc[ltv_missing_mask, "lifetime_value_imputed"].describe()
})
display(ltv_imputation_summary.round(2))

Imputed LTV values: 2,782
Remaining missing LTV: 0
Negative *predictions*: 0
Recorded negative LTV retained: 82


,Observed LTV,Imputed LTV
count,12668.00,2782.00
mean,4070.48,5084.81
std,5519.68,5953.00
min,-13440.24,18.41
25%,343.52,445.21
50%,1596.34,2836.02
75%,5903.90,7724.13
max,42139.17,28275.43


In [206]:
# Identify the origin of each missing LTV value
ltv_group = pd.Series("Observed", index=df.index)
ltv_group.loc[df["flag_ltv_imputed"]] = "Originally missing"
ltv_group.loc[df["flag_ltv_structural_missing"]] = "Zero + recency 999"
ltv_group.loc[df["flag_ltv_additional_zero_missing"]] = "Other zero"
# Compare financial predictions and customer characteristics
ltv_group_comparison = df.assign(ltv_group=ltv_group).groupby("ltv_group").agg(
    members=("member_id", "count"),
    median_ltv=("lifetime_value_imputed", "median"),
    mean_ltv=("lifetime_value_imputed", "mean"),
    median_tenure=("membership_months_clean", "median"),
    median_visits=("total_visits", "median"),
    median_renewals=("n_renewals", "median")
)
display(ltv_group_comparison.round(2))
# Compare LTV distributions across membership plans
plan_ltv_comparison = df.assign(ltv_group=ltv_group).pivot_table(
    index="plan_tier",
    columns="ltv_group",
    values="lifetime_value_imputed",
    aggfunc="median"
)
display(plan_ltv_comparison.round(2))

,members,median_ltv,mean_ltv,median_tenure,median_visits,median_renewals
ltv_group,,,,,,
Observed,12668,1596.34,4070.48,8.54,34.00,0.00
Originally missing,1040,4094.01,6148.60,16.18,85.50,1.00
Other zero,446,4456.14,6403.10,17.07,90.50,1.00
Zero + recency 999,1296,864.19,3777.48,4.19,14.00,0.00


ltv_group,Observed,Originally missing,Other zero,Zero + recency 999
plan_tier,,,,
dedicated_desk,10962.90,10771.40,11943.04,10281.25
flex,1805.42,2796.44,2775.77,592.89
full_time,5722.77,6613.76,7281.50,5354.19
hot_desk,351.39,1098.04,1169.51,312.98
private_office,21342.52,21917.59,21688.13,20991.35


**Lifetime Value Imputation Validation**
The updated MAE and RMSE are reported by the comparison cells above. The original financial values and all imputation flags are preserved, including negative amounts that may represent refunds.
**Decision:** Retain Random Forest only if it remains the best evaluated method after rerunning the notebook. Since missing LTV is potentially systematic, evaluate the impact of the imputed values and preserved negatives during Value clustering.

In [207]:
# Select records with known age
age_known = df.loc[df["age"].notna()]
age_train_idx, age_valid_idx = train_test_split(age_known.index, test_size=0.20, random_state=42)
age_train = df.loc[age_train_idx]
age_valid = df.loc[age_valid_idx]
y_age_train = age_train["age"].to_numpy(dtype=float)
y_age_valid = age_valid["age"].to_numpy(dtype=float)
# Evaluate median-based methods
age_median = age_train["age"].median()
age_plan_medians = age_train.groupby("plan_tier")["age"].median()
age_pred_global = np.full(len(age_valid), age_median)
age_pred_plan = age_valid["plan_tier"].map(age_plan_medians).fillna(age_median).to_numpy(dtype=float)
# Prepare Random Forest predictors
age_numeric = ["membership_months_clean", "total_visits", "n_renewals", "avg_weekly_visits", "n_colleagues_on_account"]
age_categorical = ["gender", "country", "industry", "plan_tier"]
age_features = age_numeric + age_categorical
age_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), age_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), age_categorical)
])
age_rf = Pipeline([
    ("preprocessor", age_preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1))
])
age_rf.fit(age_train[age_features], y_age_train)
age_pred_rf = age_rf.predict(age_valid[age_features])
# Compare all methods
age_results = pd.DataFrame([
    {"Method": "Global median", "MAE": mean_absolute_error(y_age_valid, age_pred_global), "RMSE": np.sqrt(np.mean((y_age_valid - age_pred_global) ** 2))},
    {"Method": "Median by plan", "MAE": mean_absolute_error(y_age_valid, age_pred_plan), "RMSE": np.sqrt(np.mean((y_age_valid - age_pred_plan) ** 2))},
    {"Method": "Random Forest", "MAE": mean_absolute_error(y_age_valid, age_pred_rf), "RMSE": np.sqrt(np.mean((y_age_valid - age_pred_rf) ** 2))}
])
print(f"Training observations: {len(age_train):,}")
print(f"Validation observations: {len(age_valid):,}")
display(age_results.sort_values("MAE").round(2))

Training observations: 11,800
Validation observations: 2,950


,Method,MAE,RMSE
0,Global median,6.73,8.49
1,Median by plan,6.74,8.49
2,Random Forest,6.82,8.57


**Age Imputation**

Global Median, Median by Plan and Random Forest were evaluated using the same validation observations.
Global Median achieved the lowest MAE (6.73 years) and RMSE (8.49 years), although differences between methods were small. Predictive imputation provided no measurable improvement.

**Decision:** 

Impute the 700 missing age values using the median of all observed ages. Preserve original values and record imputed observations through a quality flag.

In [208]:
# Preserve original age values
df["age_original"] = df["age"].copy()
# Identify missing ages
df["flag_age_imputed"] = df["age"].isna()
# Calculate median using all observed ages
age_final_median = df["age"].median()
# Create imputed age column
df["age_imputed"] = df["age"].fillna(age_final_median)
# Verify imputation
print(f"Median age used: {age_final_median:.1f}")
print(f"Imputed ages: {df['flag_age_imputed'].sum():,}")
print(f"Remaining missing ages: {df['age_imputed'].isna().sum():,}")
display(df[["age_original", "age_imputed"]].describe().round(2))

Median age used: 34.0
Imputed ages: 700
Remaining missing ages: 0


,age_original,age_imputed
count,14750.00,15450.00
mean,34.75,34.71
std,8.39,8.20
min,18.00,18.00
25%,29.00,29.00
50%,34.00,34.00
75%,40.00,40.00
max,93.00,93.00


**Colleagues on Account Imputation**

The variable `n_colleagues_on_account` contains 565 missing values following the removal of invalid negative entries.
Global Median, Median by Team Account status and Random Forest will be compared using the same held-out observations. Predictions are rounded to non-negative integers, and MAE and RMSE are used to evaluate performance.

In [209]:
# Select records with known colleague counts
col_known = df.loc[df["n_colleagues_on_account"].notna()]
col_train_idx, col_valid_idx = train_test_split(col_known.index, test_size=0.20, random_state=42)
col_train = df.loc[col_train_idx]
col_valid = df.loc[col_valid_idx]
y_col_train = col_train["n_colleagues_on_account"].to_numpy(dtype=float)
y_col_valid = col_valid["n_colleagues_on_account"].to_numpy(dtype=float)
# Calculate median-based predictions
col_median = col_train["n_colleagues_on_account"].median()
col_team_medians = col_train.groupby("is_team_account")["n_colleagues_on_account"].median()
col_pred_global = np.full(len(col_valid), col_median)
col_pred_team = col_valid["is_team_account"].map(col_team_medians).fillna(col_median).to_numpy(dtype=float)
# Select predictors
col_numeric = ["age_imputed", "plan_days_per_week", "total_visits", "n_renewals", "avg_weekly_visits"]
col_categorical = ["is_team_account", "plan_tier", "industry"]
col_features = col_numeric + col_categorical
# Prepare preprocessing and Random Forest
col_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), col_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), col_categorical)
])
col_rf = Pipeline([
    ("preprocessor", col_preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1))
])
col_rf.fit(col_train[col_features], y_col_train)
col_pred_rf = col_rf.predict(col_valid[col_features])
# Round predictions to non-negative integers
predictions = {
    "Global median": col_pred_global,
    "Median by team status": col_pred_team,
    "Random Forest": col_pred_rf
}
col_results = pd.DataFrame([
    {
        "Method": method,
        "MAE": mean_absolute_error(y_col_valid, np.maximum(0, np.rint(pred))),
        "RMSE": np.sqrt(np.mean((y_col_valid - np.maximum(0, np.rint(pred))) ** 2))
    }
    for method, pred in predictions.items()
])
print(f"Training observations: {len(col_train):,}")
print(f"Validation observations: {len(col_valid):,}")
display(col_results.sort_values("MAE").round(2))

Training observations: 11,908
Validation observations: 2,977


,Method,MAE,RMSE
1,Median by team status,0.28,0.69
2,Random Forest,0.28,0.70
0,Global median,0.61,1.34


**Colleagues on Account Imputation**

Median by Team Status achieved the lowest validation errors (MAE: 0.28; RMSE: 0.69), performing similarly to Random Forest and substantially better than Global Median.
**Decision:** Impute the 565 missing values using the median colleague count for each team-account category. Predictions are rounded to non-negative integers, and original values are preserved.

In [210]:
# Identify missing colleague counts
df["flag_colleagues_imputed"] = df["n_colleagues_on_account"].isna()
# Calculate team-specific medians using all observed values
col_global_median = df["n_colleagues_on_account"].median()
col_team_medians_final = df.groupby("is_team_account")["n_colleagues_on_account"].median()
print("Median colleagues by team status:")
display(col_team_medians_final.to_frame("Median colleagues"))
# Preserve original values in a separate imputed column
df["n_colleagues_on_account_imputed"] = df["n_colleagues_on_account"].copy()
# Predict missing values using team-specific medians
missing_col = df["flag_colleagues_imputed"]
col_predictions = df.loc[missing_col, "is_team_account"].map(col_team_medians_final).fillna(col_global_median)
df.loc[missing_col, "n_colleagues_on_account_imputed"] = np.maximum(0, np.rint(col_predictions.to_numpy(dtype=float)))
# Verify imputation
print(f"Colleague counts imputed: {missing_col.sum():,}")
print(f"Remaining missing values: {df['n_colleagues_on_account_imputed'].isna().sum():,}")
print(f"Negative values: {(df['n_colleagues_on_account_imputed'] < 0).sum():,}")
display(pd.crosstab(df.loc[missing_col, "is_team_account"], df.loc[missing_col, "n_colleagues_on_account_imputed"]))

Median colleagues by team status:


,Median colleagues
is_team_account,
False,0.00
True,2.00


Colleague counts imputed: 565
Remaining missing values: 0
Negative values: 0


n_colleagues_on_account_imputed,0.00,2.00
is_team_account,,
False,412,0
True,0,153


**Gender Missing Value Treatment**

Gender contains 530 missing observations (3.43%). Assigning the most frequent category or predicting gender from customer characteristics could introduce demographic bias and misrepresent the original data.

**Decision:** 

Preserve missing gender information using a separate `Not recorded` category. Keep the original values unchanged and use gender primarily for segment profiling rather than clustering.

In [211]:
# Preserve original gender values
df["gender_original"] = df["gender"].copy()
# Identify missing gender records
df["flag_gender_missing"] = df["gender"].isna()
# Create a separate category for missing information
df["gender_clean"] = df["gender"].fillna("Not recorded")
# Verify the treatment
print(f"Missing gender values categorized: {df['flag_gender_missing'].sum():,}")
print(f"Remaining missing values: {df['gender_clean'].isna().sum():,}")
display(df["gender_clean"].value_counts(dropna=False).to_frame("Members"))

Missing gender values categorized: 530
Remaining missing values: 0


,Members
gender_clean,
Female,7858
Male,6565
Not recorded,530
Other,497


**Country Missing Value Treatment**

Country contains 429 missing observations (2.78%).
Imputing the most frequent country could artificially increase the representation of Portugal and distort customer profiling.

**Decision:**

 Preserve missing country information as `Not recorded`, keeping it distinct from existing geographical categories. The original values remain unchanged.

In [212]:
# Preserve original country values
df["country_original"] = df["country"].copy()
# Flag missing country values
df["flag_country_missing"] = df["country"].isna()
# Create a separate category for missing information
df["country_clean"] = df["country"].fillna("Not recorded")
# Verify the treatment
print(f"Missing country values categorized: {df['flag_country_missing'].sum():,}")
print(f"Remaining missing values: {df['country_clean'].isna().sum():,}")
display(df["country_clean"].value_counts(dropna=False).to_frame("Members"))

Missing country values categorized: 429
Remaining missing values: 0


,Members
country_clean,
Portugal,10499
United Kingdom,1165
Germany,901
Spain,741
France,533
Netherlands,487
Other (Europe),451
Not recorded,429
Ireland,244


**Membership Plan Imputation**

The `plan_tier` variable contains 105 missing observations (0.68%). Since membership plans are closely related to customer activity and entitlements, predictive imputation may recover useful information.
Global Mode, Mode by Plan Days per Week and Random Forest will be evaluated using the same stratified validation split. Accuracy and Macro F1 are used to assess performance across membership categories.

In [213]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
# Select records with known membership plans
plan_known = df.loc[df["plan_tier"].notna()]
plan_train_idx, plan_valid_idx = train_test_split(plan_known.index, test_size=0.20, random_state=42, stratify=plan_known["plan_tier"])
plan_train = df.loc[plan_train_idx]
plan_valid = df.loc[plan_valid_idx]
y_plan_train = plan_train["plan_tier"]
y_plan_valid = plan_valid["plan_tier"]
# Global mode baseline
plan_global_mode = y_plan_train.mode().iloc[0]
plan_pred_global = np.full(len(plan_valid), plan_global_mode)
# Mode by plan days per week
plan_days_modes = plan_train.groupby("plan_days_per_week")["plan_tier"].agg(lambda x: x.mode().iloc[0])
plan_pred_days = plan_valid["plan_days_per_week"].map(plan_days_modes).fillna(plan_global_mode)
# Select predictors without using LTV-derived features
plan_numeric = ["plan_days_per_week", "entitled_visits_last_period", "actual_visits_last_period", "avg_weekly_visits", "total_visits", "membership_months_clean", "n_colleagues_on_account_imputed"]
plan_categorical = ["is_team_account", "industry"]
plan_features = plan_numeric + plan_categorical
# Prepare preprocessing
plan_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), plan_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), plan_categorical)
])
# Train Random Forest classifier
plan_rf = Pipeline([
    ("preprocessor", plan_preprocessor),
    ("model", RandomForestClassifier(n_estimators=150, min_samples_leaf=3, random_state=42, n_jobs=-1))
])
plan_rf.fit(plan_train[plan_features], y_plan_train)
plan_pred_rf = plan_rf.predict(plan_valid[plan_features])
# Compare classification methods
plan_predictions = {
    "Global mode": plan_pred_global,
    "Mode by plan days": plan_pred_days,
    "Random Forest": plan_pred_rf
}
plan_results = pd.DataFrame([
    {"Method": method, "Accuracy": accuracy_score(y_plan_valid, pred), "Macro F1": f1_score(y_plan_valid, pred, average="macro")}
    for method, pred in plan_predictions.items()
])
print(f"Training observations: {len(plan_train):,}")
print(f"Validation observations: {len(plan_valid):,}")
display(plan_results.sort_values("Macro F1", ascending=False).round(4))

Training observations: 12,276
Validation observations: 3,069


,Method,Accuracy,Macro F1
2,Random Forest,0.93,0.88
1,Mode by plan days,0.89,0.75
0,Global mode,0.36,0.10


**Membership Plan Imputation Results**

Random Forest achieved the highest Accuracy (93%) and Macro F1 (0.88), outperforming both mode-based approaches.

**Decision:**

 Retrain Random Forest using all observed membership plans and predict the 105 missing values. Preserve the original categories and record prediction confidence to identify uncertain imputations.

In [214]:
# Identify observed and missing membership plans
plan_missing_mask = df["plan_tier"].isna()
# Preserve original values
df["plan_tier_original"] = df["plan_tier"].copy()
df["flag_plan_tier_imputed"] = plan_missing_mask.copy()
# Retrain using all observed membership plans
plan_rf.fit(
    df.loc[~plan_missing_mask, plan_features],
    df.loc[~plan_missing_mask, "plan_tier"]
)
# Create imputed membership plan column
df["plan_tier_imputed"] = df["plan_tier"].copy()
df.loc[plan_missing_mask, "plan_tier_imputed"] = plan_rf.predict(df.loc[plan_missing_mask, plan_features])
# Record prediction confidence
df["plan_tier_imputation_confidence"] = np.nan
df.loc[plan_missing_mask, "plan_tier_imputation_confidence"] = plan_rf.predict_proba(df.loc[plan_missing_mask, plan_features]).max(axis=1)
# Verify imputation
print(f"Imputed membership plans: {plan_missing_mask.sum():,}")
print(f"Remaining missing plans: {df['plan_tier_imputed'].isna().sum():,}")
print(f"Predictions below 70% confidence: {(df.loc[plan_missing_mask, 'plan_tier_imputation_confidence'] < 0.70).sum():,}")
display(df.loc[plan_missing_mask, "plan_tier_imputed"].value_counts().to_frame("Imputed members"))
display(df.loc[plan_missing_mask, "plan_tier_imputation_confidence"].describe().to_frame("Prediction confidence"))

Imputed membership plans: 105
Remaining missing plans: 0
Predictions below 70% confidence: 37


,Imputed members
plan_tier_imputed,
private_office,56
dedicated_desk,34
full_time,15


,Prediction confidence
count,105.00
mean,0.81
std,0.18
min,0.45
25%,0.66
50%,0.87
75%,0.99
max,1.00


In [215]:
from sklearn.metrics import classification_report
# Evaluate performance separately for each membership plan
print("Validation performance by plan:")
print(classification_report(y_plan_valid, plan_pred_rf, digits=3, zero_division=0))
# Identify low-confidence imputed plans
low_confidence = df["flag_plan_tier_imputed"] & df["plan_tier_imputation_confidence"].lt(0.70)
print(f"Low-confidence predictions: {low_confidence.sum():,}")
display(df.loc[low_confidence, "plan_tier_imputed"].value_counts().to_frame("Members"))
# Inspect the least confident predictions
review_columns = [
    "member_id", "plan_tier_imputed", "plan_tier_imputation_confidence",
    "plan_days_per_week", "entitled_visits_last_period",
    "actual_visits_last_period", "is_team_account"
]
display(df.loc[low_confidence, review_columns].sort_values("plan_tier_imputation_confidence").head(20))
# Compare entitlement patterns for observed and imputed plans
df["plan_source"] = np.where(df["flag_plan_tier_imputed"], "Imputed", "Observed")
display(df.groupby(["plan_source", "plan_tier_imputed"])[
    ["plan_days_per_week", "entitled_visits_last_period"]
].median().round(2))

Validation performance by plan:
                precision    recall  f1-score   support

dedicated_desk      0.640     0.671     0.655       310
          flex      0.992     1.000     0.996      1090
     full_time      0.787     0.757     0.772       494
      hot_desk      1.000     0.996     0.998      1042
private_office      1.000     0.992     0.996       133

      accuracy                          0.926      3069
     macro avg      0.884     0.883     0.883      3069
  weighted avg      0.927     0.926     0.926      3069

Low-confidence predictions: 37


,Members
plan_tier_imputed,
dedicated_desk,19
full_time,15
private_office,3


,member_id,plan_tier_imputed,plan_tier_imputation_confidence,plan_days_per_week,entitled_visits_last_period,actual_visits_last_period,is_team_account
3285,CW003187,private_office,0.45,7.00,3.04,0,False
14427,CW014009,private_office,0.47,7.00,19.77,1,False
9723,CW009452,full_time,0.50,5.00,43.48,34,False
10226,CW009939,dedicated_desk,0.50,5.00,43.48,42,False
8074,CW007844,dedicated_desk,0.51,5.00,18.51,12,False
5618,CW005461,dedicated_desk,0.52,5.00,43.48,24,True
7564,CW007354,full_time,0.52,5.00,43.48,2,False
6047,CW005880,full_time,0.52,5.00,43.48,23,True
3473,CW003366,full_time,0.53,5.00,43.48,2,False
3566,CW003456,dedicated_desk,0.53,5.00,43.48,3,False


plan_days_per_week  entitled_visits_last_period
plan_source plan_tier_imputed                                                 
Imputed     dedicated_desk                   5.00                        43.48
            full_time                        5.00                        43.48
            private_office                   7.00                        60.87
Observed    dedicated_desk                   5.00                        43.48
            flex                             3.00                        26.09
            full_time                        5.00                        43.48
            hot_desk                         2.00                        17.39
            private_office                   7.00                        60.87

**Membership Plan Imputation Validation**

Random Forest achieved 92.6% validation accuracy and a Macro F1 of 0.883. Performance was strongest for Flex, Hot Desk and Private Office, while Dedicated Desk and Full-time were more difficult to distinguish due to overlapping entitlement patterns.
All 105 missing plans were imputed, with 37 predictions below 70% confidence.

**Decision:**

 Retain the predicted categories, preserve uncertainty flags and assess their influence during clustering sensitivity analysis.

In [216]:
# Preserve uncertainty information from plan tier imputation
df["flag_plan_tier_low_confidence"] = df["flag_plan_tier_imputed"] & df["plan_tier_imputation_confidence"].lt(0.70)
# Check plan days against originally observed membership plans
observed_plans = df.loc[df["plan_tier"].notna()]
print("Plan days per week by originally observed membership plan:")
display(pd.crosstab(observed_plans["plan_tier"], observed_plans["plan_days_per_week"], dropna=False))
# Examine missing plan days
missing_days = df["plan_days_per_week"].isna()
print(f"Missing plan days: {missing_days.sum():,}")
print(f"Missing both original plan tier and plan days: {(missing_days & df['flag_plan_tier_imputed']).sum():,}")
print("Membership plans for records with missing plan days:")
display(df.loc[missing_days, "plan_tier_imputed"].value_counts(dropna=False).to_frame("Members"))

Plan days per week by originally observed membership plan:


plan_days_per_week,2.00,3.00,5.00,7.00,NaN
plan_tier,,,,,
dedicated_desk,0,0,1540,0,9
flex,0,5407,0,0,42
full_time,0,0,2456,0,15
hot_desk,5183,0,0,0,29
private_office,0,0,0,662,2


Missing plan days: 97
Missing both original plan tier and plan days: 0
Membership plans for records with missing plan days:


,Members
plan_tier_imputed,
flex,42
hot_desk,29
full_time,15
dedicated_desk,9
private_office,2


**Plan Days per Week Imputation**

The relationship between membership plan and weekly entitlement is fully consistent across all observed records: Hot Desk (2 days), Flex (3 days), Full-time and Dedicated Desk (5 days) and Private Office (7 days).
None of the 97 records with missing plan days has a missing membership plan.

**Decision:** 

Recover missing values directly from the observed plan-specific entitlements, rather than using statistical or predictive imputation.

In [217]:
# Preserve original values and identify missing plan days
df["plan_days_per_week_original"] = df["plan_days_per_week"].copy()
df["flag_plan_days_imputed"] = df["plan_days_per_week"].isna()
# Derive the entitlement mapping from originally observed membership plans
plan_days_mapping = df.groupby("plan_tier")["plan_days_per_week"].median()
print("Plan days per week by membership plan:")
display(plan_days_mapping.to_frame("Days per week"))
# Create a completed column while preserving observed values
df["plan_days_per_week_imputed"] = df["plan_days_per_week"].copy()
missing_days = df["flag_plan_days_imputed"]
df.loc[missing_days, "plan_days_per_week_imputed"] = df.loc[missing_days, "plan_tier"].map(plan_days_mapping)
# Verify the recovered values
print(f"Plan days recovered: {missing_days.sum():,}")
print(f"Remaining missing values: {df['plan_days_per_week_imputed'].isna().sum():,}")
print(f"Original non-missing values changed: {(df.loc[~missing_days, 'plan_days_per_week_imputed'] != df.loc[~missing_days, 'plan_days_per_week']).sum():,}")
display(pd.crosstab(df.loc[missing_days, "plan_tier"], df.loc[missing_days, "plan_days_per_week_imputed"]))

Plan days per week by membership plan:


,Days per week
plan_tier,
dedicated_desk,5.00
flex,3.00
full_time,5.00
hot_desk,2.00
private_office,7.00


Plan days recovered: 97
Remaining missing values: 0
Original non-missing values changed: 0


plan_days_per_week_imputed,2.00,3.00,5.00,7.00
plan_tier,,,,
dedicated_desk,0,0,9,0
flex,0,42,0,0
full_time,0,0,15,0
hot_desk,29,0,0,0
private_office,0,0,0,2


**Entitled Visits Imputation Strategy**
As established during data understanding, entitlement values differ within a membership plan; the plan category is not a deterministic reconstruction rule. Compare Global Median, Median by Membership Plan and Random Forest using held-out observations for the 10 missing entitlements.

In [218]:
# Select records with observed entitlements and original membership plans
ent_known = df.loc[df["entitled_visits_last_period"].notna() & df["plan_tier"].notna()]
ent_train_idx, ent_valid_idx = train_test_split(ent_known.index, test_size=0.20, random_state=42)
ent_train = df.loc[ent_train_idx]
ent_valid = df.loc[ent_valid_idx]
y_ent_train = ent_train["entitled_visits_last_period"]
y_ent_valid = ent_valid["entitled_visits_last_period"]
# Calculate median-based predictions
ent_global_median = y_ent_train.median()
ent_plan_medians = ent_train.groupby("plan_tier")["entitled_visits_last_period"].median()
ent_pred_global = np.full(len(ent_valid), ent_global_median)
ent_pred_plan = ent_valid["plan_tier"].map(ent_plan_medians).fillna(ent_global_median)
# Select predictors without using LTV or the target itself
ent_numeric = ["plan_days_per_week_imputed", "actual_visits_last_period", "total_visits", "avg_weekly_visits", "membership_months_clean", "n_renewals"]
ent_categorical = ["plan_tier", "is_team_account"]
ent_features = ent_numeric + ent_categorical
# Prepare preprocessing
ent_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), ent_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), ent_categorical)
])
# Train Random Forest regressor
ent_rf = Pipeline([
    ("preprocessor", ent_preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1))
])
ent_rf.fit(ent_train[ent_features], y_ent_train)
ent_pred_rf = ent_rf.predict(ent_valid[ent_features])
# Compare methods using the same validation observations
ent_predictions = {
    "Global median": ent_pred_global,
    "Median by membership plan": ent_pred_plan,
    "Random Forest": ent_pred_rf
}
ent_results = pd.DataFrame([
    {
        "Method": method,
        "MAE": mean_absolute_error(y_ent_valid, pred),
        "RMSE": np.sqrt(np.mean((y_ent_valid - np.asarray(pred)) ** 2))
    }
    for method, pred in ent_predictions.items()
])
print(f"Training observations: {len(ent_train):,}")
print(f"Validation observations: {len(ent_valid):,}")
display(ent_results.sort_values("MAE").round(2))

Training observations: 12,268
Validation observations: 3,067


,Method,MAE,RMSE
2,Random Forest,0.28,2.77
1,Median by membership plan,1.96,6.09
0,Global median,10.23,13.66


**Entitled Visits Imputation Results**

Random Forest achieved the lowest validation errors (MAE: 0.28; RMSE: 2.77), substantially outperforming Global Median and Median by Membership Plan. The difference between MAE and RMSE suggests occasional larger prediction errors.

**Decision:**

 Retrain Random Forest on all available observations and impute the 10 missing entitlements. Preserve original values and flag imputed records.

In [219]:
# Preserve original values and identify missing entitlements
df["entitled_visits_last_period_original"] = df["entitled_visits_last_period"].copy()
df["flag_entitlements_imputed"] = df["entitled_visits_last_period"].isna()
ent_missing_mask = df["flag_entitlements_imputed"]
# Retrain Random Forest using all records with observed entitlements and membership plans
ent_training_mask = df["entitled_visits_last_period"].notna() & df["plan_tier"].notna()
ent_rf.fit(df.loc[ent_training_mask, ent_features], df.loc[ent_training_mask, "entitled_visits_last_period"])
# Create completed entitlement column
df["entitled_visits_last_period_imputed"] = df["entitled_visits_last_period"].copy()
ent_predictions_missing = ent_rf.predict(df.loc[ent_missing_mask, ent_features])
df.loc[ent_missing_mask, "entitled_visits_last_period_imputed"] = np.maximum(0, np.round(ent_predictions_missing, 2))
# Verify the imputation
print(f"Entitlements imputed: {ent_missing_mask.sum():,}")
print(f"Remaining missing values: {df['entitled_visits_last_period_imputed'].isna().sum():,}")
print(f"Negative values: {(df['entitled_visits_last_period_imputed'] < 0).sum():,}")
print(f"Original non-missing values changed: {(df.loc[~ent_missing_mask, 'entitled_visits_last_period_imputed'] != df.loc[~ent_missing_mask, 'entitled_visits_last_period']).sum():,}")
# Inspect all 10 imputed records
display(df.loc[ent_missing_mask, ["member_id", "plan_tier", "plan_days_per_week_imputed", "actual_visits_last_period", "entitled_visits_last_period_imputed"]])

Entitlements imputed: 10
Remaining missing values: 0
Negative values: 0
Original non-missing values changed: 0


,member_id,plan_tier,plan_days_per_week_imputed,actual_visits_last_period,entitled_visits_last_period_imputed
1988,CW001931,hot_desk,2.00,6,17.39
2381,CW002313,full_time,5.00,9,43.48
2382,CW002314,flex,3.00,26,27.75
4695,CW004557,flex,3.00,21,26.09
6973,CW006783,private_office,7.00,9,60.87
7862,CW007640,flex,3.00,23,26.09
10654,CW010353,hot_desk,2.00,2,17.15
11708,CW011372,full_time,5.00,0,43.48
12684,CW012315,hot_desk,2.00,10,14.50
13866,CW013463,full_time,5.00,0,43.48


**Event Space Hours Imputation**

Eight event-space-hour values remain missing after the earlier service-flag corrections. The EDA recommends assigning zero only when `uses_event_space` explicitly indicates no use.

**Decision:** 

Preserve the original values. For missing amounts where usage is recorded as true, use the median positive usage among recorded users; for explicitly false usage, use zero. For an unknown usage flag, use the median of comparable members with missing total visits and record uncertainty. Report all imputation counts and keep an imputation flag.

In [220]:
# Preserve original event-space usage and identify missing values
df["event_space_hours_original"] = df["event_space_hours"].copy()
df["flag_event_space_imputed"] = df["event_space_hours"].isna()
event_missing = df["flag_event_space_imputed"]
event_used = event_missing & df["uses_event_space"].eq(True)
event_not_used = event_missing & df["uses_event_space"].eq(False)
event_unknown = event_missing & df["uses_event_space"].isna()
# Estimate positive consumption only from customers recorded as event-space users
positive_user_hours = df.loc[df["uses_event_space"].eq(True) & df["event_space_hours"].gt(0), "event_space_hours"]
median_positive_usage = positive_user_hours.median()
# Unknown usage falls back to the observed median for records with missing total visits
peer_median = df.loc[df["total_visits"].isna(), "event_space_hours"].median()
df["event_space_hours_imputed"] = df["event_space_hours"].copy()
df.loc[event_not_used, "event_space_hours_imputed"] = 0
if event_used.any() and pd.isna(median_positive_usage):
    raise ValueError("Cannot estimate event-space hours for recorded users: no observed positive usage.")
df.loc[event_used, "event_space_hours_imputed"] = median_positive_usage
df.loc[event_unknown, "event_space_hours_imputed"] = peer_median
df["flag_event_space_usage_unknown"] = event_unknown
print(f"Imputed for recorded users: {event_used.sum():,}")
print(f"Imputed for explicit non-use: {event_not_used.sum():,}")
print(f"Imputed with unknown usage: {event_unknown.sum():,}")
print(f"Remaining missing hours: {df['event_space_hours_imputed'].isna().sum():,}")
print(f"Negative hours: {df['event_space_hours_imputed'].lt(0).sum():,}")
print(f"Non-missing original values changed: {(df.loc[~event_missing, 'event_space_hours_imputed'] != df.loc[~event_missing, 'event_space_hours']).sum():,}")
display(df.loc[event_missing, ["member_id", "uses_event_space", "event_space_hours_imputed"]])

Imputed for recorded users: 8
Imputed for explicit non-use: 0
Imputed with unknown usage: 0
Remaining missing hours: 0
Negative hours: 0
Non-missing original values changed: 0


,member_id,uses_event_space,event_space_hours_imputed
2748,CW002665,True,23.20
8090,CW007860,True,23.20
8760,CW008510,True,23.20
10403,CW010111,True,23.20
11346,CW011019,True,23.20
14325,CW013908,True,23.20
14347,CW013929,True,23.20
14434,CW014016,True,23.20


**Total Visits Imputation Strategy**

`total_visits` contains 208 missing values, including records flagged for inconsistent visit counts. The EDA established tenure and recent activity as useful relationships.

**Decision:**

 Validate the duration-based reconstruction against observed records before comparing it with plan-based median and predictive imputation. Preserve the original visit totals and consistency flags.

In [221]:
# Prepare observed records for validation
visits_known = df.loc[df["total_visits"].notna()].copy()
visits_train_idx, visits_valid_idx = train_test_split(visits_known.index, test_size=0.20, random_state=42)
visits_train = df.loc[visits_train_idx]
visits_valid = df.loc[visits_valid_idx]
y_visits_train = visits_train["total_visits"]
y_visits_valid = visits_valid["total_visits"]
# Establish a median-by-plan baseline
visits_global_median = y_visits_train.median()
visits_plan_medians = visits_train.groupby("plan_tier_imputed")["total_visits"].median()
visits_pred_plan = visits_valid["plan_tier_imputed"].map(visits_plan_medians).fillna(visits_global_median).to_numpy()
# Reconstruct visits from weekly frequency and membership duration
visits_pred_reconstructed = visits_valid["avg_weekly_visits"] * visits_valid["membership_months_clean"] * (52 / 12)
visits_pred_reconstructed = visits_pred_reconstructed.fillna(visits_valid["plan_tier_imputed"].map(visits_plan_medians)).fillna(visits_global_median).to_numpy()
# Select behavioral predictors for Random Forest
visits_numeric = ["avg_weekly_visits", "actual_visits_last_period", "membership_months_clean", "plan_days_per_week_imputed", "n_renewals", "days_since_last_visit", "meeting_room_hours", "guest_passes_issued", "event_space_hours_imputed"]
visits_categorical = ["plan_tier_imputed", "is_team_account"]
visits_features = visits_numeric + visits_categorical
visits_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), visits_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), visits_categorical)
])
# Train Random Forest on recorded total visits
visits_rf = Pipeline([
    ("preprocessor", visits_preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1))
])
visits_rf.fit(visits_train[visits_features], y_visits_train)
visits_pred_rf = visits_rf.predict(visits_valid[visits_features])
# Compare methods, enforcing the minimum represented by recent visits
visits_predictions = {
    "Median by membership plan": visits_pred_plan,
    "Weekly activity reconstruction": visits_pred_reconstructed,
    "Random Forest": visits_pred_rf
}
visits_results = pd.DataFrame([
    {
        "Method": method,
        "MAE": mean_absolute_error(y_visits_valid, np.maximum(visits_valid["actual_visits_last_period"].fillna(0).to_numpy(), np.maximum(0, np.rint(pred)))),
        "RMSE": np.sqrt(np.mean((y_visits_valid - np.maximum(visits_valid["actual_visits_last_period"].fillna(0).to_numpy(), np.maximum(0, np.rint(pred)))) ** 2))
    }
    for method, pred in visits_predictions.items()
])
print(f"Training observations: {len(visits_train):,}")
print(f"Validation observations: {len(visits_valid):,}")
display(visits_results.sort_values("MAE").round(2))

Training observations: 12,193
Validation observations: 3,049


,Method,MAE,RMSE
1,Weekly activity reconstruction,2.98,70.67
2,Random Forest,3.31,41.16
0,Median by membership plan,86.47,149.65


In [222]:
# Check whether weekly activity almost reproduces recorded total visits
check_mask = visits_valid["membership_months_clean"].gt(0) & visits_valid["avg_weekly_visits"].notna()
check_data = visits_valid.loc[check_mask]
reconstructed = check_data["avg_weekly_visits"] * check_data["membership_months_clean"] * (52 / 12)
reconstruction_error = (check_data["total_visits"] - reconstructed).abs()
print(f"Records checked: {len(check_data):,}")
print(f"Median absolute difference: {reconstruction_error.median():.2f}")
print(f"Within 1 visit: {(reconstruction_error <= 1).mean() * 100:.2f}%")
print(f"Within 5 visits: {(reconstruction_error <= 5).mean() * 100:.2f}%")
print(f"Missing total visits with unavailable duration: {(df['total_visits'].isna() & df['membership_months_clean'].isna()).sum():,}")

Records checked: 3,021
Median absolute difference: 0.40
Within 1 visit: 82.75%
Within 5 visits: 99.64%
Missing total visits with unavailable duration: 2


**Total Visits Imputation**

The EDA identified strong relationships between total visits, membership duration and weekly activity. Validation confirmed that reconstructing visits from average weekly activity and membership duration produces estimates within 5 visits of the recorded total for 99.64% of evaluated observations. This relationship may be partly mathematical rather than independently predictive.

**Decision:**

 Reconstruct missing total visits where valid membership duration is available, using Random Forest as a fallback. Round estimates to non-negative integers, ensure they are not below recorded visits in the last period and preserve the original values and imputation flags.

In [223]:
# Preserve original total visits and identify missing records
df["total_visits_original"] = df["total_visits"].copy()
df["flag_total_visits_imputed"] = df["total_visits"].isna()
# Identify records eligible for duration-based reconstruction
reconstruction_mask = df["flag_total_visits_imputed"] & df["membership_months_clean"].gt(0) & df["avg_weekly_visits"].notna()
fallback_mask = df["flag_total_visits_imputed"] & ~reconstruction_mask
# Preserve the method used for each imputed record
df["flag_total_visits_reconstructed"] = reconstruction_mask
df["flag_total_visits_rf_fallback"] = fallback_mask
# Create the completed total-visits column
df["total_visits_imputed"] = df["total_visits"].copy()
# Reconstruct visits using weekly activity and membership duration
reconstructed_visits = df.loc[reconstruction_mask, "avg_weekly_visits"] * df.loc[reconstruction_mask, "membership_months_clean"] * (52 / 12)
df.loc[reconstruction_mask, "total_visits_imputed"] = np.maximum(0, np.rint(reconstructed_visits))
# Retrain Random Forest using all observed total visits
visits_rf.fit(df.loc[~df["flag_total_visits_imputed"], visits_features], df.loc[~df["flag_total_visits_imputed"], "total_visits"])
# Predict records where reconstruction is unavailable
if fallback_mask.any():
    fallback_predictions = visits_rf.predict(df.loc[fallback_mask, visits_features])
    df.loc[fallback_mask, "total_visits_imputed"] = np.maximum(0, np.rint(fallback_predictions))
# Ensure imputed lifetime visits are not below recorded recent visits
missing_mask = df["flag_total_visits_imputed"]
recent_visits = df.loc[missing_mask, "actual_visits_last_period"].fillna(0)
df.loc[missing_mask, "total_visits_imputed"] = np.maximum(df.loc[missing_mask, "total_visits_imputed"], recent_visits)
# Verify the final results
print(f"Total visits imputed: {missing_mask.sum():,}")
print(f"Reconstructed from weekly activity: {reconstruction_mask.sum():,}")
print(f"Random Forest fallback: {fallback_mask.sum():,}")
print(f"Remaining missing values: {df['total_visits_imputed'].isna().sum():,}")
print(f"Negative values: {(df['total_visits_imputed'] < 0).sum():,}")
print(f"Original non-missing values changed: {(df.loc[~missing_mask, 'total_visits_imputed'] != df.loc[~missing_mask, 'total_visits']).sum():,}")
display(df.loc[missing_mask, "total_visits_imputed"].describe().round(2).to_frame("Imputed total visits"))

Total visits imputed: 208
Reconstructed from weekly activity: 206
Random Forest fallback: 2
Remaining missing values: 0
Negative values: 0
Original non-missing values changed: 0


,Imputed total visits
count,208.00
mean,97.80
std,187.18
min,2.00
25%,12.00
50%,24.00
75%,108.25
max,1794.00


**Meeting Room Hours and Guest Passes Imputation**

The EDA identified inconsistencies between service-usage flags and recorded quantities. Missing quantities associated with explicit non-use were previously replaced with zero.

**Decision strategy:**

 Assess the remaining missing values using their corresponding usage flags and estimate missing consumption only where appropriate, preserving original observations and imputation flags.

In [224]:
# Assess the remaining missing service quantities using their usage flags
service_pairs = {
    "meeting_room_hours": "uses_meeting_rooms",
    "guest_passes_issued": "uses_guest_passes"
}
service_results = []
for quantity, flag in service_pairs.items():
    missing = df[quantity].isna()
    recorded_users = df.loc[df[flag].eq(True) & df[quantity].notna(), quantity]
    positive_usage = recorded_users[recorded_users > 0]
    service_results.append({
        "Variable": quantity,
        "Missing": missing.sum(),
        "Missing - usage True": (missing & df[flag].eq(True)).sum(),
        "Missing - usage False": (missing & df[flag].eq(False)).sum(),
        "Missing - usage Unknown": (missing & df[flag].isna()).sum(),
        "Median among recorded users": recorded_users.median(),
        "Median among positive users": positive_usage.median(),
        "Zero usage among recorded users (%)": (recorded_users.eq(0).mean() * 100)
    })
service_results = pd.DataFrame(service_results).set_index("Variable")
display(service_results.round(2))

,Missing,Missing - usage True,Missing - usage False,Missing - usage Unknown,Median among recorded users,Median among positive users,Zero usage among recorded users (%)
Variable,,,,,,,
meeting_room_hours,409,409,0,0,33.20,33.30,0.14
guest_passes_issued,210,210,0,0,18.00,20.00,4.77


**Meeting Room Hours and Guest Passes Imputation**

All remaining missing service quantities correspond to customers with confirmed usage flags. Consequently, replacing these observations with zero would be inconsistent with the recorded service information.

**Decision:** 

Impute missing values using the median positive consumption within each membership plan, with the overall positive-user median as a fallback. Preserve original values and imputation flags. Guest-pass estimates are rounded to non-negative integers.

In [225]:
# Define service quantities and their corresponding usage flags
service_pairs = {
    "meeting_room_hours": "uses_meeting_rooms",
    "guest_passes_issued": "uses_guest_passes"
}
# Apply plan-specific median imputation to both variables
for quantity, flag in service_pairs.items():
    missing = df[quantity].isna()
    if not df.loc[missing, flag].eq(True).all():
        raise ValueError(f"Unexpected missing values without confirmed usage: {quantity}")
    # Calculate medians using customers with confirmed positive consumption
    reference = df.loc[df[flag].eq(True) & df[quantity].gt(0)]
    plan_medians = reference.groupby("plan_tier_imputed")[quantity].median()
    global_median = reference[quantity].median()
    # Preserve original values and identify imputed records
    df[f"{quantity}_original"] = df[quantity].copy()
    df[f"flag_{quantity}_imputed"] = missing.copy()
    df[f"{quantity}_imputed"] = df[quantity].copy()
    # Estimate missing consumption using membership-plan medians
    estimates = df.loc[missing, "plan_tier_imputed"].map(plan_medians).fillna(global_median)
    if quantity == "guest_passes_issued":
        estimates = np.maximum(1, np.rint(estimates))
    else:
        estimates = estimates.round(2).clip(lower=0)
    df.loc[missing, f"{quantity}_imputed"] = estimates
    # Verify the imputation
    print(f"\n{quantity}")
    print(f"Values imputed: {missing.sum():,}")
    print(f"Remaining missing: {df[f'{quantity}_imputed'].isna().sum():,}")
    print(f"Negative values: {(df[f'{quantity}_imputed'] < 0).sum():,}")
    print(f"Original non-missing values changed: {(df.loc[~missing, f'{quantity}_imputed'] != df.loc[~missing, quantity]).sum():,}")
    print(f"Median of imputed values: {df.loc[missing, f'{quantity}_imputed'].median():.2f}")


meeting_room_hours
Values imputed: 409
Remaining missing: 0
Negative values: 0
Original non-missing values changed: 0
Median of imputed values: 20.30

guest_passes_issued
Values imputed: 210
Remaining missing: 0
Negative values: 0
Original non-missing values changed: 0
Median of imputed values: 12.00


**Number of Renewals Imputation**

The `n_renewals` variable contains 66 missing observations. Membership duration and customer activity may provide useful information for estimating renewal counts.

**Decision strategy:** 

Compare Global Median, Median by Membership Duration and Random Forest using held-out observations. Predictions will be rounded to non-negative integers, with original values and imputation flags preserved.

In [226]:
# Select records with observed renewal counts
renew_known = df.loc[df["n_renewals"].notna()]
renew_train_idx, renew_valid_idx = train_test_split(renew_known.index, test_size=0.20, random_state=42)
renew_train = df.loc[renew_train_idx]
renew_valid = df.loc[renew_valid_idx]
y_renew_train = renew_train["n_renewals"]
y_renew_valid = renew_valid["n_renewals"]
# Calculate the global median baseline
renew_global_median = y_renew_train.median()
renew_pred_global = np.full(len(renew_valid), renew_global_median)
# Compare customers within membership-duration groups
renew_bins = [0, 3, 6, 12, 24, np.inf]
renew_train_groups = pd.cut(renew_train["membership_months_clean"], bins=renew_bins, include_lowest=True)
renew_valid_groups = pd.cut(renew_valid["membership_months_clean"], bins=renew_bins, include_lowest=True)
renew_duration_medians = y_renew_train.groupby(renew_train_groups, observed=True).median()
renew_pred_duration = renew_valid_groups.map(renew_duration_medians).astype(float).fillna(renew_global_median).to_numpy()
# Select behavioral predictors without using LTV
renew_numeric = ["membership_months_clean", "total_visits_imputed", "avg_weekly_visits", "actual_visits_last_period", "age_imputed", "plan_days_per_week_imputed", "days_since_last_visit"]
renew_categorical = ["plan_tier_imputed", "is_team_account"]
renew_features = renew_numeric + renew_categorical
# Build the Random Forest preprocessing pipeline
renew_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), renew_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), renew_categorical)
])
renew_rf = Pipeline([
    ("preprocessor", renew_preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1))
])
renew_rf.fit(renew_train[renew_features], y_renew_train)
renew_pred_rf = renew_rf.predict(renew_valid[renew_features])
# Compare predictions as non-negative integer counts
renew_predictions = {
    "Global median": renew_pred_global,
    "Median by membership duration": renew_pred_duration,
    "Random Forest": renew_pred_rf
}
renew_results = pd.DataFrame([
    {
        "Method": method,
        "MAE": mean_absolute_error(y_renew_valid, np.maximum(0, np.rint(pred))),
        "RMSE": np.sqrt(np.mean((y_renew_valid - np.maximum(0, np.rint(pred))) ** 2))
    }
    for method, pred in renew_predictions.items()
])
print(f"Training observations: {len(renew_train):,}")
print(f"Validation observations: {len(renew_valid):,}")
display(renew_results.sort_values("MAE").round(3))

Training observations: 12,307
Validation observations: 3,077


,Method,MAE,RMSE
2,Random Forest,0.63,1.04
1,Median by membership duration,0.71,1.20
0,Global median,1.05,1.73


**Number of Renewals Imputation Results**

Random Forest achieved the lowest validation errors (MAE: 0.63; RMSE: 1.04), outperforming the duration-based median and global median.

**Decision:**

 Retrain Random Forest using all observed renewal counts and impute the 66 missing values. Predictions are rounded to non-negative integers, with original values and imputation flags preserved.

In [227]:
# Preserve original renewal counts and identify missing values
df["n_renewals_original"] = df["n_renewals"].copy()
df["flag_renewals_imputed"] = df["n_renewals"].isna()
renew_missing_mask = df["flag_renewals_imputed"]
# Retrain Random Forest using all observed renewal counts
renew_rf.fit(df.loc[~renew_missing_mask, renew_features], df.loc[~renew_missing_mask, "n_renewals"])
# Create the completed renewal count column
df["n_renewals_imputed"] = df["n_renewals"].copy()
renew_missing_predictions = renew_rf.predict(df.loc[renew_missing_mask, renew_features])
df.loc[renew_missing_mask, "n_renewals_imputed"] = np.maximum(0, np.rint(renew_missing_predictions))
# Verify the imputation
print(f"Renewal counts imputed: {renew_missing_mask.sum():,}")
print(f"Remaining missing values: {df['n_renewals_imputed'].isna().sum():,}")
print(f"Negative values: {(df['n_renewals_imputed'] < 0).sum():,}")
print(f"Original non-missing values changed: {(df.loc[~renew_missing_mask, 'n_renewals_imputed'] != df.loc[~renew_missing_mask, 'n_renewals']).sum():,}")
print("Distribution of imputed renewal counts:")
display(df.loc[renew_missing_mask, "n_renewals_imputed"].value_counts().sort_index().to_frame("Members"))

Renewal counts imputed: 66
Remaining missing values: 0
Negative values: 0
Original non-missing values changed: 0
Distribution of imputed renewal counts:


,Members
n_renewals_imputed,
0.00,19
1.00,27
2.00,11
3.00,9


**Remaining Recency Imputation**

After reconstructing 1,267 missing recency values from reliable dates, 87 observations remain unresolved due to temporal inconsistencies.

**Decision strategy:** 

Compare Global Median, Median by Membership Status and Random Forest using held-out recency observations. Preserve the original values, reconstructed values and temporal reliability flags. Estimated recency values will not be treated as verified dates.

In [228]:
# Select records with known or reliably reconstructed recency
rec_known = df.loc[df["days_since_last_visit"].notna()]
rec_train_idx, rec_valid_idx = train_test_split(rec_known.index, test_size=0.20, random_state=42)
rec_train = df.loc[rec_train_idx]
rec_valid = df.loc[rec_valid_idx]
y_rec_train = rec_train["days_since_last_visit"]
y_rec_valid = rec_valid["days_since_last_visit"]
# Calculate simple baseline predictions
rec_global_median = y_rec_train.median()
rec_status_medians = rec_train.groupby("is_active")["days_since_last_visit"].median()
rec_pred_global = np.full(len(rec_valid), rec_global_median)
rec_pred_status = rec_valid["is_active"].map(rec_status_medians).fillna(rec_global_median).to_numpy()
# Select available behavioral predictors, excluding unreliable dates
rec_numeric = ["membership_months_clean", "total_visits_imputed", "avg_weekly_visits", "actual_visits_last_period", "n_renewals_imputed", "plan_days_per_week_imputed", "age_imputed", "is_active"]
rec_categorical = ["plan_tier_imputed", "industry"]
rec_features = rec_numeric + rec_categorical
# Prepare the Random Forest pipeline
rec_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), rec_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), rec_categorical)
])
rec_rf = Pipeline([
    ("preprocessor", rec_preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1))
])
rec_rf.fit(rec_train[rec_features], y_rec_train)
rec_pred_rf = rec_rf.predict(rec_valid[rec_features])
# Compare methods using non-negative whole days
rec_predictions = {
    "Global median": rec_pred_global,
    "Median by membership status": rec_pred_status,
    "Random Forest": rec_pred_rf
}
rec_results = pd.DataFrame([
    {
        "Method": method,
        "MAE": mean_absolute_error(y_rec_valid, np.maximum(0, np.rint(pred))),
        "RMSE": np.sqrt(np.mean((y_rec_valid - np.maximum(0, np.rint(pred))) ** 2))
    }
    for method, pred in rec_predictions.items()
])
print(f"Training observations: {len(rec_train):,}")
print(f"Validation observations: {len(rec_valid):,}")
display(rec_results.sort_values("MAE").round(2))

Training observations: 12,290
Validation observations: 3,073


,Method,MAE,RMSE
2,Random Forest,15.10,37.80
1,Median by membership status,34.37,71.08
0,Global median,34.58,73.01


**Remaining Recency Imputation Results**

Random Forest achieved the lowest validation errors (MAE: 15.10 days; RMSE: 37.80 days), substantially outperforming both median-based approaches.

**Decision:**

 Retrain Random Forest on all available recency observations and estimate the remaining 87 missing values. Predictions are rounded to non-negative whole days. A separate flag identifies statistically imputed values, which are less reliable than recency reconstructed from valid dates.

In [229]:
# Preserve recency values before statistical imputation
df["days_since_last_visit_before_rf"] = df["days_since_last_visit"].copy()
# Identify the remaining missing recency values
rec_missing_mask = df["days_since_last_visit"].isna()
df["flag_recency_rf_imputed"] = rec_missing_mask.copy()
# Retrain Random Forest using all available recency observations
rec_rf.fit(df.loc[~rec_missing_mask, rec_features], df.loc[~rec_missing_mask, "days_since_last_visit"])
# Create the completed recency column
df["days_since_last_visit_imputed"] = df["days_since_last_visit"].copy()
# Predict missing values and round to non-negative whole days
rec_missing_predictions = rec_rf.predict(df.loc[rec_missing_mask, rec_features])
df.loc[rec_missing_mask, "days_since_last_visit_imputed"] = np.maximum(0, np.rint(rec_missing_predictions))
# Verify the imputation
print(f"Recency values imputed: {rec_missing_mask.sum():,}")
print(f"Remaining missing values: {df['days_since_last_visit_imputed'].isna().sum():,}")
print(f"Negative values: {(df['days_since_last_visit_imputed'] < 0).sum():,}")
print(f"Original non-missing values changed: {(df.loc[~rec_missing_mask, 'days_since_last_visit_imputed'] != df.loc[~rec_missing_mask, 'days_since_last_visit']).sum():,}")
print("Distribution of statistically imputed recency:")
display(df.loc[rec_missing_mask, "days_since_last_visit_imputed"].describe().round(2).to_frame("Imputed recency (days)"))

Recency values imputed: 87
Remaining missing values: 0
Negative values: 0
Original non-missing values changed: 0
Distribution of statistically imputed recency:


,Imputed recency (days)
count,87.00
mean,48.84
std,61.93
min,3.00
25%,6.50
50%,34.00
75%,46.50
max,225.00


**Membership Duration Reconstruction**

The remaining 152 missing membership durations result from unresolved chronological inconsistencies. Since total visits and average weekly visits are strongly related to membership duration, an activity-based reconstruction will be evaluated against reliable observed durations.
Any recovered values will represent estimated membership duration, not corrected membership dates.

In [230]:
# Calculate an estimated membership duration from recorded activity
weeks_per_month = 30.44 / 7
duration_estimate = df["total_visits_imputed"] / (df["avg_weekly_visits"] * weeks_per_month)
duration_estimate = duration_estimate.replace([np.inf, -np.inf], np.nan).where(lambda x: x.gt(0))
# Validate against existing reliable membership durations
duration_valid = df["membership_months_clean"].gt(0) & duration_estimate.notna()
duration_error = (df.loc[duration_valid, "membership_months_clean"] - duration_estimate.loc[duration_valid]).abs()
print(f"Reliable durations evaluated: {duration_valid.sum():,}")
print(f"Median absolute error (months): {duration_error.median():.2f}")
print(f"90th percentile absolute error: {duration_error.quantile(0.90):.2f}")
print(f"Within 1 month: {(duration_error <= 1).mean() * 100:.2f}%")
# Assess whether the 152 missing durations can be reconstructed
duration_missing = df["membership_months_clean"].isna()
duration_recoverable = duration_missing & duration_estimate.notna()
print(f"Missing membership durations: {duration_missing.sum():,}")
print(f"Potentially recoverable: {duration_recoverable.sum():,}")
print(f"Still unresolved: {(duration_missing & ~duration_recoverable).sum():,}")
display(duration_estimate.loc[duration_recoverable].describe().round(2).to_frame("Estimated duration (months)"))

Reliable durations evaluated: 15,284
Median absolute error (months): 0.05
90th percentile absolute error: 0.30
Within 1 month: 97.42%
Missing membership durations: 152
Potentially recoverable: 152
Still unresolved: 0


,Estimated duration (months)
count,152.00
mean,12.70
std,11.36
min,0.06
25%,2.14
50%,9.71
75%,22.27
max,40.24


**Membership Duration Reconstruction Results**

Activity-based reconstruction was evaluated against 15,284 reliable membership durations, achieving a median absolute error of 0.05 months, with 97.42% of estimates within one month.
All 152 missing durations have sufficient information for reconstruction. However, the relationship may be partly mathematical because average weekly visits and membership duration are related.

**Decision:** 

Reconstruct the 152 missing durations using total visits and average weekly visits. Preserve the original values, unreliable date flags and reconstruction indicators. Two estimates rely on previously imputed total visits and are flagged separately. Reconstructed durations do not imply that the original dates have been corrected.

In [231]:
# Preserve the original cleaned duration and identify missing values
df["membership_months_before_reconstruction"] = df["membership_months_clean"].copy()
duration_missing = df["membership_months_clean"].isna()
# Keep separate flags for reconstructed durations and less reliable inputs
df["flag_membership_months_reconstructed"] = duration_missing.copy()
df["flag_membership_months_rf_visits_input"] = duration_missing & df["flag_total_visits_rf_fallback"]
# Create the completed duration column without modifying original dates
df["membership_months_imputed"] = df["membership_months_clean"].copy()
df.loc[duration_missing, "membership_months_imputed"] = duration_estimate.loc[duration_missing].round(2)
# Validate the reconstruction
print(f"Durations reconstructed: {duration_missing.sum():,}")
print(f"Using Random Forest-imputed visits: {df['flag_membership_months_rf_visits_input'].sum():,}")
print(f"Remaining missing durations: {df['membership_months_imputed'].isna().sum():,}")
print(f"Non-positive durations: {(df['membership_months_imputed'] <= 0).sum():,}")
print(f"Original valid durations changed: {(df.loc[~duration_missing, 'membership_months_imputed'] != df.loc[~duration_missing, 'membership_months_clean']).sum():,}")
display(df.loc[duration_missing, "membership_months_imputed"].describe().round(2).to_frame("Reconstructed membership months"))

Durations reconstructed: 152
Using Random Forest-imputed visits: 2
Remaining missing durations: 0
Non-positive durations: 0
Original valid durations changed: 0


,Reconstructed membership months
count,152.00
mean,12.71
std,11.36
min,0.06
25%,2.14
50%,9.71
75%,22.27
max,40.24


## 4. Feature Engineering
Feature engineering follows the findings and recommendations established during the EDA. The objective is to construct meaningful indicators of customer engagement, service consumption and economic value using cleaned and imputed variables.
### 4.1. Engagement and Utilization
Three indicators are created to describe customer engagement:
- `utilization_rate_last_period`: Proportion of entitled visits actually used.
- `visit_intensity_gap`: Difference between average weekly visits and permitted weekly visits.
- `recency_ratio`: Time since the last visit relative to membership duration.

In [232]:
# Calculate the proportion of entitled visits actually used
df["utilization_rate_last_period"] = (
    df["actual_visits_last_period"] /
    df["entitled_visits_last_period_imputed"].where(
        df["entitled_visits_last_period_imputed"] > 0
    )
)
# Measure the difference between weekly activity and plan entitlement
df["visit_intensity_gap"] = (
    df["avg_weekly_visits"] - df["plan_days_per_week_imputed"]
)
# Express recency relative to the cleaned or reconstructed membership duration
membership_duration_days = df["membership_months_imputed"] * 30.44
df["recency_ratio"] = (
    df["days_since_last_visit_imputed"] /
    membership_duration_days.where(membership_duration_days > 0)
)
# Verify that the engineered features contain valid numerical values
engagement_features = [
    "utilization_rate_last_period",
    "visit_intensity_gap",
    "recency_ratio"
]
print("Missing values:")
display(df[engagement_features].isna().sum().to_frame("Missing"))
print("Feature distributions:")
display(df[engagement_features].describe().round(2))
print(f"Utilization above 100%: {(df['utilization_rate_last_period'] > 1).sum():,}")

Missing values:


,Missing
utilization_rate_last_period,0
visit_intensity_gap,0
recency_ratio,0


Feature distributions:


,utilization_rate_last_period,visit_intensity_gap,recency_ratio
count,15450.00,15450.00,15450.00
mean,0.52,-1.60,0.30
std,0.31,2.07,0.42
min,0.00,-6.65,0.00
25%,0.24,-2.02,0.01
50%,0.52,-1.24,0.06
75%,0.80,-0.63,0.54
max,1.00,42.40,14.24


Utilization above 100%: 0


### 4.2. Service Usage and Add-on Diversity
The `addon_diversity_score` measures the number of different add-on services used by each customer. It captures the breadth of service engagement rather than the quantity consumed.

In [233]:
# Define the nine add-on services identified in the EDA
addon_cols = [
    "uses_meeting_rooms",
    "uses_event_space",
    "uses_guest_passes",
    "uses_print",
    "uses_mail_handling",
    "uses_phone_booths",
    "uses_day_passes",
    "uses_storage",
    "uses_bike_locker"
]
# Verify that all service flags are present and contain valid binary values
assert all(col in df.columns for col in addon_cols), "Some add-on columns are missing."
assert df[addon_cols].isin([True, False, 0, 1]).all().all(), "Invalid or missing add-on flags detected."
# Count the number of distinct add-on services used by each customer
df["addon_diversity_score"] = df[addon_cols].astype(int).sum(axis=1)
# Verify the engineered feature
print(f"Missing values: {df['addon_diversity_score'].isna().sum():,}")
print(f"Minimum add-on diversity: {df['addon_diversity_score'].min()}")
print(f"Maximum add-on diversity: {df['addon_diversity_score'].max()}")
print("Distribution of add-on diversity:")
display(df["addon_diversity_score"].value_counts().sort_index().to_frame("Customers"))

Missing values: 0
Minimum add-on diversity: 0
Maximum add-on diversity: 8
Distribution of add-on diversity:


,Customers
addon_diversity_score,
0,3087
1,4643
2,3236
3,2139
4,1395
5,690
6,223
7,34
8,3


### 4.3. Service Usage Intensity
Cumulative service consumption is normalized by membership duration to improve comparability between customers:
- `meeting_room_hours_per_month`: Average monthly meeting-room usage.
- `event_space_hours_per_month`: Average monthly event-space usage.
- `guest_passes_per_month`: Average monthly guest passes issued.

In [234]:
# Define cleaned service quantities
service_intensity_cols = {
    "meeting_room_hours_imputed": "meeting_room_hours_per_month",
    "event_space_hours_imputed": "event_space_hours_per_month",
    "guest_passes_issued_imputed": "guest_passes_per_month"
}
# Use cleaned and reconstructed membership duration
duration_months = df["membership_months_imputed"].where(df["membership_months_imputed"] > 0)
# Calculate monthly service consumption rates
for source_col, new_col in service_intensity_cols.items():
    df[new_col] = df[source_col] / duration_months
# Verify the engineered features
service_intensity_features = list(service_intensity_cols.values())
print("Missing values:")
display(df[service_intensity_features].isna().sum().to_frame("Missing"))
print("Non-finite values:")
display((~np.isfinite(df[service_intensity_features])).sum().to_frame("Non-finite"))
print("Feature distributions:")
display(df[service_intensity_features].describe().round(2))

Missing values:


C:\Users\Mister\AppData\Local\Temp\ipykernel_9692\2360661412.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df[source_col] / duration_months


,Missing
meeting_room_hours_per_month,0
event_space_hours_per_month,0
guest_passes_per_month,0


Non-finite values:


,Non-finite
meeting_room_hours_per_month,0
event_space_hours_per_month,0
guest_passes_per_month,0


Feature distributions:


,meeting_room_hours_per_month,event_space_hours_per_month,guest_passes_per_month
count,15450.00,15450.00,15450.00
mean,2.06,0.33,0.40
std,21.05,1.50,2.08
min,0.00,0.00,0.00
25%,0.00,0.00,0.00
50%,0.00,0.00,0.00
75%,1.01,0.00,0.00
max,1842.49,27.10,182.64


### 4.4. Economic and Value Features
Three indicators are calculated to capture different dimensions of customer value:
- `revenue_per_visit`: Lifetime value divided by total visits.
- `value_per_member`: Lifetime value divided by the number of account members.
- `lifetime_value_per_month`: Lifetime value normalized by membership duration.

In [235]:
# Calculate the total number of people sharing each account
df["total_account_members"] = df["n_colleagues_on_account_imputed"] + 1
# Calculate revenue generated per recorded visit
df["revenue_per_visit"] = (
    df["lifetime_value_imputed"] /
    df["total_visits_imputed"].where(df["total_visits_imputed"] > 0)
)
# Calculate lifetime value adjusted for account size
df["value_per_member"] = (
    df["lifetime_value_imputed"] /
    df["total_account_members"].where(df["total_account_members"] > 0)
)
# Normalize lifetime value by membership duration
df["lifetime_value_per_month"] = (
    df["lifetime_value_imputed"] /
    df["membership_months_imputed"].where(df["membership_months_imputed"] > 0)
)
# Verify the engineered economic features
value_features = ["revenue_per_visit", "value_per_member", "lifetime_value_per_month"]
print("Missing values:")
display(df[value_features].isna().sum().to_frame("Missing"))
print("Non-finite values:")
display((~np.isfinite(df[value_features])).sum().to_frame("Non-finite"))
print("Feature distributions:")
display(df[value_features].describe().round(2))
print(f"Memberships shorter than 1 month: {(df['membership_months_imputed'] < 1).sum():,}")
print(f"Negative monthly values: {(df['lifetime_value_per_month'] < 0).sum():,}")

Missing values:


C:\Users\Mister\AppData\Local\Temp\ipykernel_9692\3119583296.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["total_account_members"] = df["n_colleagues_on_account_imputed"] + 1
C:\Users\Mister\AppData\Local\Temp\ipykernel_9692\3119583296.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["revenue_per_visit"] = (
C:\Users\Mister\AppData\Local\Temp\ipykernel_9692\3119583296.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

,Missing
revenue_per_visit,0
value_per_member,0
lifetime_value_per_month,0


Non-finite values:


,Non-finite
revenue_per_visit,0
value_per_member,0
lifetime_value_per_month,0


Feature distributions:


,revenue_per_visit,value_per_member,lifetime_value_per_month
count,15450.00,15450.00,15450.00
mean,129.33,3474.22,269.99
std,530.25,4990.78,234.83
min,-872.02,-13440.24,-4100.38
25%,20.69,297.21,150.95
50%,31.04,1186.43,207.72
75%,74.02,4674.74,331.12
max,14030.39,42139.17,15615.35


Memberships shorter than 1 month: 702
Negative monthly values: 82


### 4.5. Membership Plan Encoding
Membership plans are encoded according to their predefined hierarchy, from Hot Desk (1) to Private Office (5). The original categorical values are preserved.
This encoding assumes an ordered relationship between plans; whether to include it in the final clustering features will be determined during feature selection.

In [236]:
# Define the membership plan hierarchy
plan_mapping = {
    "hot_desk": 1,
    "flex": 2,
    "dedicated_desk": 3,
    "full_time": 4,
    "private_office": 5
}
# Encode membership plans using the completed categorical column
df["plan_tier_encoded"] = df["plan_tier_imputed"].map(plan_mapping)
# Verify that every membership plan was successfully encoded
print(f"Missing encoded values: {df['plan_tier_encoded'].isna().sum():,}")
print("Membership plan encoding:")
display(pd.crosstab(df["plan_tier_imputed"], df["plan_tier_encoded"]))

Missing encoded values: 0
Membership plan encoding:


C:\Users\Mister\AppData\Local\Temp\ipykernel_9692\1861376482.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["plan_tier_encoded"] = df["plan_tier_imputed"].map(plan_mapping)


plan_tier_encoded,1,2,3,4,5
plan_tier_imputed,,,,,
dedicated_desk,0,0,1583,0,0
flex,0,5449,0,0,0
full_time,0,0,0,2486,0
hot_desk,5212,0,0,0,0
private_office,0,0,0,0,720


## 5. Feature Transformations and Outlier Treatment
Highly skewed features and extreme observations can disproportionately influence distance-based clustering. Transformations and outlier treatment will be applied before final scaling, while preserving the original measurements.
### 5.1. Skewness Transformations
Logarithmic transformations (`log1p`) are applied to non-negative activity measures, while the inverse hyperbolic sine (`arcsinh`) is used for financial indicators containing negative values. The transformed features are stored separately for subsequent feature selection.

In [237]:
# Define non-negative features for logarithmic transformation
log_features = [
    "meeting_room_hours_per_month",
    "event_space_hours_per_month",
    "guest_passes_per_month",
    "recency_ratio"
]
# Define financial features that may contain negative values
signed_features = [
    "revenue_per_visit",
    "value_per_member",
    "lifetime_value_per_month"
]
# Verify that logarithmic inputs are non-negative
assert df[log_features].ge(0).all().all(), "Negative values detected in log transformation inputs."
# Create transformed features without modifying original columns
log_transformed = np.log1p(df[log_features]).add_suffix("_log1p")
signed_transformed = np.arcsinh(df[signed_features]).add_suffix("_asinh")
transformed_features = pd.concat([log_transformed, signed_transformed], axis=1)
# Add transformed columns together to avoid DataFrame fragmentation
df = df.drop(columns=transformed_features.columns, errors="ignore").join(transformed_features)
# Verify transformations
print(f"Transformed features created: {len(transformed_features.columns)}")
print(f"Missing values: {df[transformed_features.columns].isna().sum().sum():,}")
print(f"Non-finite values: {(~np.isfinite(df[transformed_features.columns])).sum().sum():,}")
print("Transformed feature ranges:")
display(df[transformed_features.columns].agg(["min", "median", "max"]).T.round(2))

Transformed features created: 7
Missing values: 0
Non-finite values: 0
Transformed feature ranges:


,min,median,max
meeting_room_hours_per_month_log1p,0.00,0.00,7.52
event_space_hours_per_month_log1p,0.00,0.00,3.34
guest_passes_per_month_log1p,0.00,0.00,5.21
recency_ratio_log1p,0.00,0.06,2.72
revenue_per_visit_asinh,-7.46,4.13,10.24
value_per_member_asinh,-10.20,7.77,11.34
lifetime_value_per_month_asinh,-9.01,6.03,10.35


### 5.2. Outlier Assessment
Extreme values are assessed using the 1st and 99th percentiles, where applicable. Particular attention is given to customers with memberships shorter than one month, as duration-normalized features can become disproportionately large.
Outliers are retained during this assessment. Any subsequent treatment will depend on their plausibility and potential influence on clustering.

In [238]:
# Define the engineered features to assess
outlier_features = [
    "meeting_room_hours_per_month",
    "event_space_hours_per_month",
    "guest_passes_per_month",
    "recency_ratio",
    "revenue_per_visit",
    "value_per_member",
    "lifetime_value_per_month"
]
financial_features = ["revenue_per_visit", "value_per_member", "lifetime_value_per_month"]
# Identify customers with short or reconstructed membership durations
short_membership = df["membership_months_imputed"] < 1
reconstructed_membership = df["flag_membership_months_reconstructed"]
# Assess extreme values without modifying the dataset
outlier_summary = []
for col in outlier_features:
    upper = df[col].quantile(0.99)
    lower = df[col].quantile(0.01) if col in financial_features else np.nan
    extreme = df[col] > upper
    if col in financial_features:
        extreme = extreme | (df[col] < lower)
    outlier_summary.append({
        "Feature": col,
        "P1": lower,
        "P99": upper,
        "Maximum": df[col].max(),
        "Extreme records": extreme.sum(),
        "Short membership (%)": short_membership.loc[extreme].mean() * 100,
        "Reconstructed duration (%)": reconstructed_membership.loc[extreme].mean() * 100
    })
outlier_summary = pd.DataFrame(outlier_summary).set_index("Feature")
print(f"Customers with memberships shorter than 1 month: {short_membership.sum():,} ({short_membership.mean() * 100:.2f}%)")
print("Extreme-value assessment:")
display(outlier_summary.round(2))

Customers with memberships shorter than 1 month: 702 (4.54%)
Extreme-value assessment:


,P1,P99,Maximum,Extreme records,Short membership (%),Reconstructed duration (%)
Feature,,,,,,
meeting_room_hours_per_month,NaN,20.31,1842.49,155,19.35,0.00
event_space_hours_per_month,NaN,7.60,27.10,155,3.87,0.65
guest_passes_per_month,NaN,6.43,182.64,155,6.45,1.29
recency_ratio,NaN,1.02,14.24,155,6.45,8.39
revenue_per_visit,13.82,1632.00,14030.39,310,11.29,2.26
value_per_member,15.39,24291.48,42139.17,310,25.16,1.29
lifetime_value_per_month,115.45,799.09,15615.35,310,17.10,5.16


### 5.2. Outlier Assessment and Treatment
Extreme observations were assessed using the 99th percentile for non-negative variables and both the 1st and 99th percentiles for financial variables.
Customers with memberships shorter than one month represent 4.54% of the dataset but are disproportionately represented among extreme meeting-room usage and financial indicators.

**Decision:**

 Retain all observations and use the transformed features to reduce the influence of extreme values. Flag unusually high or low observations for subsequent sensitivity analysis, without applying automatic removal or winsorization.

In [239]:
# Identify extreme observations using the established percentile thresholds
outlier_flags = pd.DataFrame(index=df.index)
for col in outlier_features:
    upper = df[col].quantile(0.99)
    extreme = df[col] > upper
    if col in financial_features:
        lower = df[col].quantile(0.01)
        extreme = extreme | df[col].lt(lower)
    outlier_flags[f"flag_extreme_{col}"] = extreme
# Identify short memberships and customers with at least one extreme feature
outlier_flags["flag_short_membership"] = df["membership_months_imputed"].lt(1)
outlier_flags["flag_any_extreme_feature"] = outlier_flags.filter(like="flag_extreme_").any(axis=1)
outlier_flags["flag_short_and_extreme"] = outlier_flags["flag_short_membership"] & outlier_flags["flag_any_extreme_feature"]
# Add all flags together while avoiding DataFrame fragmentation
df = pd.concat([df.drop(columns=outlier_flags.columns, errors="ignore"), outlier_flags], axis=1).copy()
# Summarize flagged customers without excluding observations
print(f"Total customers: {len(df):,}")
print(f"Customers with at least one extreme feature: {df['flag_any_extreme_feature'].sum():,}")
print(f"Customers with short memberships: {df['flag_short_membership'].sum():,}")
print(f"Customers with short memberships and extreme features: {df['flag_short_and_extreme'].sum():,}")
print(f"Rows removed: 0")

Total customers: 15,450
Customers with at least one extreme feature: 1,152
Customers with short memberships: 702
Customers with short memberships and extreme features: 158
Rows removed: 0


## 6. Feature Selection
Feature selection begins with a systematic review of all original variables, followed by an assessment of cleaned and engineered alternatives. The objective is to retain informative, interpretable features while minimizing redundancy and avoiding unnecessary dimensions.
### 6.1. Initial Feature Inventory
All original variables are reviewed according to their data type, completeness, variability and suitability for clustering. Identifiers, dates and profiling variables are distinguished from potential clustering inputs before any features are excluded.

In [240]:
# Recover every original feature before preprocessing
original_features = df_raw.columns.tolist()
# Identify cleaned or imputed alternatives where available
replacement_suffixes = ["_imputed", "_clean"]
# Build the initial inventory
feature_inventory = pd.DataFrame({
    "Feature": original_features,
    "Data type": df_raw.dtypes.astype(str).values,
    "Missing (%)": (df_raw.isna().mean() * 100).round(2).values,
    "Unique values": df_raw.nunique(dropna=True).values
})
# Identify available cleaned versions without discarding original variables
def find_cleaned_version(col):
    alternatives = [col + suffix for suffix in replacement_suffixes]
    if col == "membership_months":
        alternatives.insert(0, "membership_months_imputed")
    available = [name for name in alternatives if name in df.columns]
    return available[0] if available else col
feature_inventory["Prepared version"] = feature_inventory["Feature"].apply(find_cleaned_version)
# Classify features for further review, without excluding them
def classify_feature(col):
    if col == "member_id":
        return "Identifier"
    if pd.api.types.is_datetime64_any_dtype(df_raw[col]) or col in ["membership_start", "membership_end"]:
        return "Date"
    if col in ["gender", "country", "age"]:
        return "Demographic / profiling"
    if pd.api.types.is_object_dtype(df_raw[col]) or isinstance(df_raw[col].dtype, pd.CategoricalDtype):
        return "Categorical candidate"
    return "Numerical / binary candidate"
feature_inventory["Initial role"] = feature_inventory["Feature"].apply(classify_feature)
print(f"Original features identified: {len(original_features)}")
print(f"Original dataset rows: {len(df_raw):,}")
print("Complete original feature inventory:")
display(feature_inventory)

Original features identified: 31
Original dataset rows: 15,465
Complete original feature inventory:


,Feature,Data type,Missing (%),Unique values,Prepared version,Initial role
0,member_id,object,0.00,15450,member_id,Identifier
1,age,float64,3.61,84,age_imputed,Demographic / profiling
2,gender,object,3.43,12,gender_clean,Demographic / profiling
3,industry,object,0.00,43,industry,Categorical candidate
4,country,object,2.78,33,country_clean,Demographic / profiling
5,is_team_account,bool,0.00,2,is_team_account,Numerical / binary candidate
6,n_colleagues_on_account,int64,0.00,11,n_colleagues_on_account_imputed,Numerical / binary candidate
7,plan_tier,object,0.68,19,plan_tier_imputed,Categorical candidate
8,plan_days_per_week,int64,0.00,6,plan_days_per_week_imputed,Numerical / binary candidate
9,membership_start,object,0.00,1539,membership_start,Date


In [241]:
# Identify all columns added during preprocessing
new_features = sorted(set(df.columns) - set(df_raw.columns))
print(f"Original features: {df_raw.shape[1]}")
print(f"Current features: {df.shape[1]}")
print(f"Columns added during preprocessing: {len(new_features)}")
# Display cleaned, imputed, engineered and transformed columns
display(pd.DataFrame({"Added column": new_features}))

Original features: 31
Current features: 148
Columns added during preprocessing: 117


,Added column
0,addon_diversity_score
1,age_imputed
2,age_original
3,country_clean
4,country_original
...,...
112,total_visits_original
113,utilization_rate_last_period
114,value_per_member
115,value_per_member_asinh


### 6.2. Feature Screening and Preparation
The initial inventory is consolidated by replacing original variables with their cleaned or imputed versions where available. Intermediate columns, diagnostic flags, identifiers and raw dates are excluded from the clustering candidate dataset.
Engineered indicators are added as alternatives to the original measurements. Final selection will consider redundancy, business relevance, categorical encoding and the suitability of each variable for clustering.

In [242]:
# Exclude identifiers and raw dates as direct clustering inputs
excluded_originals = {
    "member_id", "membership_start", "membership_end", "last_badge_swipe"
}
# Select the prepared version of every remaining original feature
prepared_features = []
for col in df_raw.columns:
    if col in excluded_originals:
        continue
    alternatives = [f"{col}_imputed", f"{col}_clean", col]
    prepared = next((name for name in alternatives if name in df.columns), None)
    if prepared is not None:
        prepared_features.append(prepared)
# Include the relevant engineered indicators
engineered_candidates = [
    "membership_months_imputed",
    "is_active",
    "total_account_members",
    "utilization_rate_last_period",
    "visit_intensity_gap",
    "recency_ratio_log1p",
    "addon_diversity_score",
    "meeting_room_hours_per_month_log1p",
    "event_space_hours_per_month_log1p",
    "guest_passes_per_month_log1p",
    "revenue_per_visit_asinh",
    "value_per_member_asinh",
    "lifetime_value_per_month_asinh"
]
# Combine original prepared variables and engineered features without duplicates
candidate_features = list(dict.fromkeys(
    prepared_features + [col for col in engineered_candidates if col in df.columns]
))
df_candidates = df[candidate_features].copy()
# Summarize the consolidated candidates
candidate_overview = pd.DataFrame({
    "Feature": df_candidates.columns,
    "Data type": df_candidates.dtypes.astype(str).values,
    "Missing (%)": (df_candidates.isna().mean() * 100).values,
    "Unique values": df_candidates.nunique(dropna=True).values
})
print(f"Original variables: {len(df_raw.columns)}")
print(f"Prepared original candidates: {len(prepared_features)}")
print(f"Total consolidated candidates: {df_candidates.shape[1]}")
print(f"Customers retained: {len(df_candidates):,}")
display(candidate_overview.round(2))

Original variables: 31
Prepared original candidates: 27
Total consolidated candidates: 40
Customers retained: 15,450


,Feature,Data type,Missing (%),Unique values
0,age_imputed,float64,0.00,57
1,gender_clean,object,0.00,4
2,industry,object,0.00,11
3,country_clean,object,0.00,9
4,is_team_account,bool,0.00,2
5,n_colleagues_on_account_imputed,float64,0.00,10
6,plan_tier_imputed,object,0.00,5
7,plan_days_per_week_imputed,float64,0.00,4
8,days_since_last_visit_imputed,float64,0.00,382
9,total_visits_imputed,float64,0.00,777


### 6.3. Initial Feature Screening
The consolidated candidates are screened according to their analytical purpose and structural redundancy. Demographic variables are reserved for cluster profiling in the primary analysis, while deterministically redundant features are excluded from the clustering candidate set.
All other variables are retained for subsequent assessment of variability, relevance and statistical redundancy.

In [243]:
# Identify demographic variables reserved for cluster profiling
profiling_features = {
    "age_imputed": "Demographic characteristic",
    "gender_clean": "Demographic characteristic",
    "country_clean": "Geographic characteristic"
}
# Check whether engineered account size duplicates existing information
account_size_redundant = (
    df_candidates["total_account_members"]
    .eq(df_candidates["n_colleagues_on_account_imputed"] + 1)
    .all()
)
# Check whether plan days are completely determined by membership plan
plan_days_redundant = (
    df_candidates.groupby("plan_tier_imputed")["plan_days_per_week_imputed"]
    .nunique().max() == 1
)
# Record initial screening decisions
screening = pd.DataFrame({"Feature": candidate_features})
screening["Decision"] = "Retain for assessment"
screening["Reason"] = "Potential clustering input"
for col, reason in profiling_features.items():
    screening.loc[screening["Feature"] == col, ["Decision", "Reason"]] = ["Profiling only", reason]
if account_size_redundant:
    screening.loc[screening["Feature"] == "total_account_members", ["Decision", "Reason"]] = [
        "Exclude - redundant", "Equivalent to colleagues on account + 1"
    ]
if plan_days_redundant:
    screening.loc[screening["Feature"] == "plan_days_per_week_imputed", ["Decision", "Reason"]] = [
        "Exclude - redundant", "Already determined by membership plan"
    ]
# Preserve the candidate dataset and create a screened feature list
screened_features = screening.loc[
    screening["Decision"] == "Retain for assessment", "Feature"
].tolist()
df_screened = df_candidates[screened_features].copy()
# Summarize decisions
print("Initial screening decisions:")
display(screening["Decision"].value_counts().to_frame("Features"))
print("\nFeatures reserved or excluded:")
display(screening[screening["Decision"] != "Retain for assessment"])
print(f"\nRemaining clustering candidates: {len(screened_features)}")
print(f"Customers retained: {len(df_screened):,}")

Initial screening decisions:


,Features
Decision,
Retain for assessment,35
Profiling only,3
Exclude - redundant,2



Features reserved or excluded:


,Feature,Decision,Reason
0,age_imputed,Profiling only,Demographic characteristic
1,gender_clean,Profiling only,Demographic characteristic
3,country_clean,Profiling only,Geographic characteristic
7,plan_days_per_week_imputed,Exclude - redundant,Already determined by membership plan
29,total_account_members,Exclude - redundant,Equivalent to colleagues on account + 1



Remaining clustering candidates: 35
Customers retained: 15,450


### 6.4. Variability and Numerical Redundancy
Numerical and binary candidates are assessed for variability and strong monotonic relationships. Features dominated by a single value are identified, while highly correlated pairs are reviewed for potential redundancy.
A correlation threshold of |ρ| ≥ 0.80 is used to flag pairs for assessment, not as an automatic exclusion rule. Categorical variables will be assessed separately.

In [244]:
# Separate numerical/binary candidates from categorical candidates
numeric_cols = df_screened.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = df_screened.select_dtypes(exclude=["number", "bool"]).columns.tolist()
numeric_data = df_screened[numeric_cols].astype(float)
print(f"Numerical/binary candidates: {len(numeric_cols)}")
print(f"Categorical candidates: {len(categorical_cols)}")
print(f"Categorical features: {categorical_cols}")
# Assess variability and identify features dominated by one value
variability = pd.DataFrame({
    "Unique values": numeric_data.nunique(),
    "Most common value (%)": numeric_data.apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100),
    "Standard deviation": numeric_data.std(ddof=0)
})
print("\nFeatures with the highest concentration of a single value:")
display(variability.sort_values("Most common value (%)", ascending=False).head(15).round(2))
# Identify strongly correlated numerical feature pairs
corr = numeric_data.corr(method="spearman")
upper_corr = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
correlation_pairs = upper_corr.stack().reset_index()
correlation_pairs.columns = ["Feature 1", "Feature 2", "Spearman correlation"]
correlation_pairs["Absolute correlation"] = correlation_pairs["Spearman correlation"].abs()
strong_pairs = correlation_pairs[
    correlation_pairs["Absolute correlation"] >= 0.80
].sort_values("Absolute correlation", ascending=False)
print(f"\nFeature pairs with |Spearman correlation| >= 0.80: {len(strong_pairs)}")
display(strong_pairs.head(25).round(3))

Numerical/binary candidates: 33
Categorical candidates: 2
Categorical features: ['industry', 'plan_tier_imputed']

Features with the highest concentration of a single value:


,Unique values,Most common value (%),Standard deviation
uses_bike_locker,2,99.83,0.04
event_space_hours_per_month_log1p,1636,88.56,0.42
event_space_hours_imputed,816,88.56,26.02
uses_event_space,2,88.10,0.32
uses_storage,2,85.16,0.36
guest_passes_issued_imputed,200,82.61,22.55
guest_passes_per_month_log1p,2054,82.61,0.44
uses_guest_passes,2,81.81,0.39
uses_day_passes,2,78.03,0.41
uses_mail_handling,2,76.96,0.42



Feature pairs with |Spearman correlation| >= 0.80: 22


,Feature 1,Feature 2,Spearman correlation,Absolute correlation
247,event_space_hours_imputed,event_space_hours_per_month_log1p,1.00,1.00
271,guest_passes_issued_imputed,guest_passes_per_month_log1p,1.00,1.00
222,meeting_room_hours_imputed,meeting_room_hours_per_month_log1p,0.99,0.99
0,is_team_account,n_colleagues_on_account_imputed,0.98,0.98
230,event_space_hours_imputed,uses_event_space,0.98,0.98
313,uses_event_space,event_space_hours_per_month_log1p,0.98,0.98
254,guest_passes_issued_imputed,uses_guest_passes,0.97,0.97
334,uses_guest_passes,guest_passes_per_month_log1p,0.97,0.97
205,meeting_room_hours_imputed,uses_meeting_rooms,0.96,0.96
291,uses_meeting_rooms,meeting_room_hours_per_month_log1p,0.96,0.96


### 6.5. Redundancy-Based Feature Screening
Strongly correlated features are reviewed according to their meaning and contribution to customer segmentation. Where multiple features represent the same behavior, a preferred representation is retained.
Monthly service-usage indicators are preferred over cumulative measures, account size is preferred over the binary team-account indicator and monthly LTV is preferred over cumulative LTV.
Rare service indicators are reserved for profiling when their inclusion could disproportionately influence distance-based clustering. All exclusions remain reversible.

In [245]:
# Define redundancy-based exclusions and their justification
redundancy_exclusions = {
    "meeting_room_hours_imputed": "Prefer monthly meeting-room usage",
    "event_space_hours_imputed": "Prefer monthly event-space usage",
    "guest_passes_issued_imputed": "Prefer monthly guest-pass usage",
    "uses_meeting_rooms": "Strong overlap with meeting-room usage intensity",
    "uses_event_space": "Strong overlap with event-space usage intensity",
    "uses_guest_passes": "Strong overlap with guest-pass usage intensity",
    "is_team_account": "Prefer number of colleagues for account structure",
    "actual_visits_last_period": "Strong overlap with weekly visits and utilization",
    "visit_intensity_gap": "Strong overlap with utilization rate",
    "lifetime_value_imputed": "Prefer tenure-normalized financial value"
}
# Update the screening decisions without deleting any columns
for feature, reason in redundancy_exclusions.items():
    mask = screening["Feature"].eq(feature)
    screening.loc[mask, ["Decision", "Reason"]] = ["Exclude - redundant", reason]
# Reserve the extremely rare bike-locker indicator for profiling
mask = screening["Feature"].eq("uses_bike_locker")
screening.loc[mask, ["Decision", "Reason"]] = [
    "Profiling only", "Very rare service usage (approximately 0.17%)"
]
# Rebuild the candidate dataset using the updated decisions
screened_features = screening.loc[
    screening["Decision"] == "Retain for assessment", "Feature"
].tolist()
df_screened = df_candidates[screened_features].copy()
# Review the revised screening results
print("Updated screening decisions:")
display(screening["Decision"].value_counts().to_frame("Features"))
print(f"\nRemaining candidates: {len(screened_features)}")
print(f"Customers retained: {len(df_screened):,}")
print("\nFeatures excluded in this stage:")
display(screening[screening["Feature"].isin(
    list(redundancy_exclusions) + ["uses_bike_locker"]
)][["Feature", "Decision", "Reason"]])

Updated screening decisions:


,Features
Decision,
Retain for assessment,24
Exclude - redundant,12
Profiling only,4



Remaining candidates: 24
Customers retained: 15,450

Features excluded in this stage:


,Feature,Decision,Reason
4,is_team_account,Exclude - redundant,Prefer number of colleagues for account structure
12,actual_visits_last_period,Exclude - redundant,Strong overlap with weekly visits and utilization
13,meeting_room_hours_imputed,Exclude - redundant,Prefer monthly meeting-room usage
14,event_space_hours_imputed,Exclude - redundant,Prefer monthly event-space usage
15,guest_passes_issued_imputed,Exclude - redundant,Prefer monthly guest-pass usage
16,uses_meeting_rooms,Exclude - redundant,Strong overlap with meeting-room usage intensity
17,uses_event_space,Exclude - redundant,Strong overlap with event-space usage intensity
18,uses_guest_passes,Exclude - redundant,Strong overlap with guest-pass usage intensity
24,uses_bike_locker,Profiling only,Very rare service usage (approximately 0.17%)
26,lifetime_value_imputed,Exclude - redundant,Prefer tenure-normalized financial value


### 6.6. Categorical Feature Assessment
The remaining categorical features are assessed according to category frequency, potential sparsity and association with one another.
Cramér's V is used to evaluate the strength of association between nominal variables. Encoding decisions will be made after determining whether these variables contribute meaningful information to clustering.

In [246]:
from scipy.stats import chi2_contingency
# Identify categorical features remaining after screening
categorical_features = df_screened.select_dtypes(exclude=["number", "bool"]).columns.tolist()
# Examine category distributions
for col in categorical_features:
    counts = df_screened[col].value_counts(dropna=False)
    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": (counts / len(df_screened) * 100).round(2)
    })
    print(f"\n{col} — {len(counts)} categories:")
    display(summary)
# Assess association between industry and membership plan
contingency = pd.crosstab(df_screened["industry"], df_screened["plan_tier_imputed"])
chi2, p_value, _, _ = chi2_contingency(contingency)
n = contingency.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contingency.shape[0] - 1, contingency.shape[1] - 1)))
print(f"\nCramér's V (Industry vs Plan Tier): {cramers_v:.3f}")
print(f"Chi-square p-value: {p_value:.4g}")


industry — 11 categories:


,Count,Percentage
industry,,
Information Technology,3557,23.02
Consulting,2055,13.30
Design & Creative,1974,12.78
Marketing & PR,1962,12.70
Finance,1219,7.89
Other,1175,7.61
Legal,878,5.68
Education,775,5.02
Unknown,693,4.49



plan_tier_imputed — 5 categories:


,Count,Percentage
plan_tier_imputed,,
flex,5449,35.27
hot_desk,5212,33.73
full_time,2486,16.09
dedicated_desk,1583,10.25
private_office,720,4.66



Cramér's V (Industry vs Plan Tier): 0.122
Chi-square p-value: 3.127e-168


### 6.7. Candidate Feature Organization
The remaining candidates are organized into engagement, service usage, economic value and account characteristics. This distinction supports feature selection within each business dimension while preserving potentially useful information from the original and engineered variables.

In [247]:
# Organize the remaining 24 candidates by business dimension
feature_groups = {
    "Engagement": [
        "days_since_last_visit_imputed",
        "total_visits_imputed",
        "avg_weekly_visits",
        "entitled_visits_last_period_imputed",
        "n_renewals_imputed",
        "membership_months_imputed",
        "is_active",
        "utilization_rate_last_period",
        "recency_ratio_log1p"
    ],
    "Service usage": [
        "uses_print",
        "uses_mail_handling",
        "uses_phone_booths",
        "uses_day_passes",
        "uses_storage",
        "addon_diversity_score",
        "meeting_room_hours_per_month_log1p",
        "event_space_hours_per_month_log1p",
        "guest_passes_per_month_log1p"
    ],
    "Economic value": [
        "revenue_per_visit_asinh",
        "value_per_member_asinh",
        "lifetime_value_per_month_asinh"
    ],
    "Account characteristics": [
        "n_colleagues_on_account_imputed",
        "plan_tier_imputed",
        "industry"
    ]
}
# Verify that every remaining candidate is accounted for exactly once
grouped_features = [col for features in feature_groups.values() for col in features]
assert len(grouped_features) == len(set(grouped_features)), "Duplicate features across groups."
assert set(grouped_features) == set(screened_features), "Candidate list does not match screening."
# Summarize the groups
group_summary = pd.DataFrame([
    {"Dimension": group, "Feature": col, "Data type": str(df_screened[col].dtype)}
    for group, features in feature_groups.items()
    for col in features
])
print("Candidates by business dimension:")
display(group_summary.groupby("Dimension").size().to_frame("Features"))
display(group_summary)

Candidates by business dimension:


,Features
Dimension,
Account characteristics,3
Economic value,3
Engagement,9
Service usage,9


,Dimension,Feature,Data type
0,Engagement,days_since_last_visit_imputed,float64
1,Engagement,total_visits_imputed,float64
2,Engagement,avg_weekly_visits,float64
3,Engagement,entitled_visits_last_period_imputed,float64
4,Engagement,n_renewals_imputed,float64
5,Engagement,membership_months_imputed,float64
6,Engagement,is_active,int64
7,Engagement,utilization_rate_last_period,float64
8,Engagement,recency_ratio_log1p,float64
9,Service usage,uses_print,int64


### 6.8. Preliminary Feature Selection
The remaining features are assigned to core clustering inputs, alternative candidates or profiling variables based on business relevance, redundancy and interpretability.
Core features establish the initial clustering configuration, while alternatives are preserved for comparisons. In particular, aggregate service diversity will be compared with individual service-use indicators, and categorical account characteristics will be evaluated separately.

In [248]:
# Assign every remaining candidate to a preliminary role
selection_roles = {
    "Core": {
        "Engagement": [
            "utilization_rate_last_period",
            "recency_ratio_log1p",
            "n_renewals_imputed"
        ],
        "Service usage": [
            "addon_diversity_score",
            "meeting_room_hours_per_month_log1p",
            "event_space_hours_per_month_log1p",
            "guest_passes_per_month_log1p"
        ],
        "Economic value": [
            "lifetime_value_per_month_asinh",
            "value_per_member_asinh"
        ]
    },
    "Alternative": {
        "Engagement": [
            "days_since_last_visit_imputed",
            "total_visits_imputed",
            "avg_weekly_visits",
            "entitled_visits_last_period_imputed",
            "membership_months_imputed"
        ],
        "Service usage": [
            "uses_print",
            "uses_mail_handling",
            "uses_phone_booths",
            "uses_day_passes",
            "uses_storage"
        ],
        "Economic value": [
            "revenue_per_visit_asinh"
        ],
        "Account characteristics": [
            "n_colleagues_on_account_imputed",
            "plan_tier_imputed",
            "industry"
        ]
    },
    "Profiling": {
        "Engagement": ["is_active"]
    }
}
# Build the preliminary selection register
selection_review = pd.DataFrame([
    {"Feature": feature, "Dimension": group, "Role": role}
    for role, groups in selection_roles.items()
    for group, features in groups.items()
    for feature in features
])
# Ensure all 24 remaining features are accounted for exactly once
assert selection_review["Feature"].is_unique, "Duplicate feature assignments detected."
assert set(selection_review["Feature"]) == set(screened_features), "Missing or unexpected features."
# Summarize the preliminary selection
print("Preliminary feature decisions:")
display(pd.crosstab(selection_review["Dimension"], selection_review["Role"]))
print(f"Total features reviewed: {len(selection_review)}")
display(selection_review.sort_values(["Dimension", "Role", "Feature"]))

Preliminary feature decisions:


Role,Alternative,Core,Profiling
Dimension,,,
Account characteristics,3,0,0
Economic value,1,2,0
Engagement,5,3,1
Service usage,5,4,0


Total features reviewed: 24


,Feature,Dimension,Role
22,industry,Account characteristics,Alternative
20,n_colleagues_on_account_imputed,Account characteristics,Alternative
21,plan_tier_imputed,Account characteristics,Alternative
19,revenue_per_visit_asinh,Economic value,Alternative
7,lifetime_value_per_month_asinh,Economic value,Core
8,value_per_member_asinh,Economic value,Core
11,avg_weekly_visits,Engagement,Alternative
9,days_since_last_visit_imputed,Engagement,Alternative
12,entitled_visits_last_period_imputed,Engagement,Alternative
13,membership_months_imputed,Engagement,Alternative


### 6.9. Categorical Encoding
The remaining categorical candidates are encoded using One-Hot Encoding, preserving their nominal meaning without imposing arbitrary numerical distances.
Encoded features are retained for alternative clustering configurations. Their contribution will be evaluated separately to avoid giving categorical dimensions disproportionate influence on distance calculations.

In [249]:
# Identify the remaining categorical candidates
categorical_cols = ["industry", "plan_tier_imputed"]
# Apply One-Hot Encoding to the categorical variables
encoded_categories = pd.get_dummies(
    df_screened[categorical_cols],
    prefix=["industry", "plan"],
    dtype=int
)
# Preserve numerical and binary candidates
numeric_candidates = df_screened.drop(columns=categorical_cols).astype(float)
# Combine numerical candidates with encoded categorical features
df_encoded_candidates = pd.concat([numeric_candidates, encoded_categories], axis=1)
# Validate the encoded dataset
assert df_encoded_candidates.index.equals(df_screened.index)
assert np.isfinite(df_encoded_candidates.to_numpy()).all()
assert encoded_categories.filter(like="industry_").sum(axis=1).eq(1).all()
assert encoded_categories.filter(like="plan_").sum(axis=1).eq(1).all()
# Summarize the encoding
print(f"Numerical/binary features: {numeric_candidates.shape[1]}")
print(f"Encoded categorical features: {encoded_categories.shape[1]}")
print(f"Total encoded candidates: {df_encoded_candidates.shape[1]}")
print(f"Customers retained: {len(df_encoded_candidates):,}")
display(encoded_categories.head())

Numerical/binary features: 22
Encoded categorical features: 16
Total encoded candidates: 38
Customers retained: 15,450


,industry_Consulting,industry_Design & Creative,industry_Education,industry_Finance,industry_Health,industry_Information Technology,industry_Legal,industry_Marketing & PR,industry_Other,industry_Real Estate,industry_Unknown,plan_dedicated_desk,plan_flex,plan_full_time,plan_hot_desk,plan_private_office
0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
2,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0


### 6.10. Clustering Feature Configurations
Core and alternative feature configurations are defined to evaluate whether different representations of engagement, service consumption and economic value produce more informative customer segments.
The configurations preserve complementary information while limiting redundancy. Encoded categorical variables are included only in selected alternatives, with their weighting to be considered during scaling and clustering.

In [250]:
# Retrieve the previously selected core features
core = selection_roles["Core"]
# Identify the encoded membership-plan columns
plan_cols = [col for col in encoded_categories.columns if col.startswith("plan_")]
# Define baseline and alternative clustering configurations
feature_sets = {
    "engagement_core": core["Engagement"].copy(),
    "engagement_frequency": [
        "avg_weekly_visits",
        "recency_ratio_log1p",
        "n_renewals_imputed"
    ],
    "service_core": core["Service usage"].copy(),
    "service_preferences": [
        "uses_print",
        "uses_mail_handling",
        "uses_phone_booths",
        "uses_day_passes",
        "uses_storage",
        "meeting_room_hours_per_month_log1p",
        "event_space_hours_per_month_log1p",
        "guest_passes_per_month_log1p"
    ],
    "economic_core": core["Economic value"].copy(),
    "economic_account_size": core["Economic value"] + [
        "n_colleagues_on_account_imputed"
    ],
    "economic_plan": core["Economic value"] + plan_cols
}
# Verify feature availability, uniqueness and numerical validity
for name, features in feature_sets.items():
    assert len(features) == len(set(features)), f"Duplicate features in {name}"
    assert set(features).issubset(df_encoded_candidates.columns), f"Missing features in {name}"
    assert np.isfinite(df_encoded_candidates[features].to_numpy(dtype=float)).all(), f"Invalid values in {name}"
# Summarize the feature configurations
configuration_summary = pd.DataFrame([
    {
        "Configuration": name,
        "Features": len(features),
        "One-hot columns": sum(col in encoded_categories.columns for col in features),
        "Customers": len(df_encoded_candidates)
    }
    for name, features in feature_sets.items()
])
display(configuration_summary)

,Configuration,Features,One-hot columns,Customers
0,engagement_core,3,0,15450
1,engagement_frequency,3,0,15450
2,service_core,4,0,15450
3,service_preferences,8,0,15450
4,economic_core,2,0,15450
5,economic_account_size,3,0,15450
6,economic_plan,7,5,15450


## 7. Feature Scaling
Feature scaling is applied separately to each clustering configuration to prepare comparable numerical inputs for distance-based algorithms.
Continuous features are standardized using `StandardScaler`, while binary service indicators retain their 0/1 representation. One-hot membership-plan variables are weighted as a group to prevent categorical information from dominating the distance calculation.
Original feature values are preserved, and fitted scalers are retained for reproducibility and subsequent sensitivity analysis.

In [251]:
from sklearn.preprocessing import StandardScaler
# Store the scaled configurations and their fitted scalers
scaled_feature_sets = {}
fitted_scalers = {}
scaling_results = []
# Identify categorical and binary columns
plan_columns = [col for col in encoded_categories.columns if col.startswith("plan_")]
# Scale every feature configuration
for name, features in feature_sets.items():
    X = df_encoded_candidates[features].astype(float)
    plan_features = [col for col in features if col in plan_columns]
    binary_features = [col for col in features if col.startswith("uses_")]
    continuous_features = [col for col in features if col not in plan_features + binary_features]
    scaled = X.copy()
    # Standardize continuous numerical features
    scaler = StandardScaler()
    scaled[continuous_features] = scaler.fit_transform(X[continuous_features])
    # Preserve binary service indicators without standardization
    scaled[binary_features] = X[binary_features]
    # Weight the membership-plan one-hot group
    if plan_features:
        scaled[plan_features] = X[plan_features] / np.sqrt(2)
    # Validate the scaled dataset
    assert scaled.index.equals(df.index), f"Index mismatch in {name}"
    assert np.isfinite(scaled.to_numpy()).all(), f"Invalid values in {name}"
    scaled_feature_sets[name] = scaled
    fitted_scalers[name] = scaler
    scaling_results.append({
        "Configuration": name,
        "Customers": len(scaled),
        "Continuous": len(continuous_features),
        "Binary": len(binary_features),
        "Plan one-hot": len(plan_features),
        "Total features": scaled.shape[1]
    })
# Display the results
scaling_summary = pd.DataFrame(scaling_results)
display(scaling_summary)

,Configuration,Customers,Continuous,Binary,Plan one-hot,Total features
0,engagement_core,15450,3,0,0,3
1,engagement_frequency,15450,3,0,0,3
2,service_core,15450,4,0,0,4
3,service_preferences,15450,3,5,0,8
4,economic_core,15450,2,0,0,2
5,economic_account_size,15450,3,0,0,3
6,economic_plan,15450,2,0,5,7


In [252]:
# Check the numerical properties of the scaled features
for name, scaled in scaled_feature_sets.items():
    print(f"\n{name}")
    display(scaled.agg(["mean", "std", "min", "max"]).T.round(3))


engagement_core


,mean,std,min,max
utilization_rate_last_period,-0.00,1.00,-1.66,1.54
recency_ratio_log1p,0.00,1.00,-0.83,9.30
n_renewals_imputed,-0.00,1.00,-0.76,3.72



engagement_frequency


,mean,std,min,max
avg_weekly_visits,0.00,1.00,-0.92,23.54
recency_ratio_log1p,0.00,1.00,-0.83,9.30
n_renewals_imputed,-0.00,1.00,-0.76,3.72



service_core


,mean,std,min,max
addon_diversity_score,-0.00,1.00,-1.21,4.11
meeting_room_hours_per_month_log1p,-0.00,1.00,-0.56,8.81
event_space_hours_per_month_log1p,0.00,1.00,-0.30,7.64
guest_passes_per_month_log1p,-0.00,1.00,-0.39,11.40


... (4 outputs left out)


economic_account_size


,mean,std,min,max
lifetime_value_per_month_asinh,-0.00,1.00,-14.96,4.22
value_per_member_asinh,0.00,1.00,-8.94,1.84
n_colleagues_on_account_imputed,0.00,1.00,-0.52,7.30



economic_plan


,mean,std,min,max
lifetime_value_per_month_asinh,-0.00,1.00,-14.96,4.22
value_per_member_asinh,0.00,1.00,-8.94,1.84
plan_dedicated_desk,0.07,0.21,0.00,0.71
plan_flex,0.25,0.34,0.00,0.71
plan_full_time,0.11,0.26,0.00,0.71
plan_hot_desk,0.24,0.33,0.00,0.71
plan_private_office,0.03,0.15,0.00,0.71


## 8. Final Validation and Data Export
The final preprocessing outputs are validated for completeness, numerical validity and consistent customer alignment.
Seven clustering configurations are exported alongside the complete preprocessed dataset, feature definitions and fitted scalers. Extreme observations are preserved for subsequent robustness and sensitivity analysis during clustering.

In [253]:
import json
import joblib
# Define the output directory
output_dir = DATA / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
# Validate all configurations before export
assert df.index.is_unique, "Customer indices are not unique."
assert len(df) == 15450, "Unexpected customer count."
for name, features in feature_sets.items():
    scaled = scaled_feature_sets[name]
    assert scaled.index.equals(df.index), f"Index mismatch: {name}"
    assert scaled.columns.tolist() == features, f"Feature mismatch: {name}"
    assert np.isfinite(scaled.to_numpy(dtype=float)).all(), f"Invalid values: {name}"
# Export the complete preprocessed dataset
df.to_csv(output_dir / "customers_preprocessed.csv", index_label="row_id")
# Export each clustering configuration
for name, scaled in scaled_feature_sets.items():
    scaled.to_csv(output_dir / f"{name}_scaled.csv", index_label="row_id")
# Save feature definitions and scaling information
metadata = {
    "feature_sets": feature_sets,
    "scaling": {
        "continuous": "StandardScaler",
        "binary": "Unscaled (0/1)",
        "plan_one_hot": "Weighted by 1/sqrt(2)"
    }
}
with open(output_dir / "clustering_features.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)
joblib.dump(fitted_scalers, output_dir / "fitted_scalers.joblib")
# Preserve the feature-selection decisions
screening.to_csv(output_dir / "feature_screening.csv", index=False)
selection_review.to_csv(output_dir / "feature_selection.csv", index=False)
print(f"Customers validated: {len(df):,}")
print(f"Clustering configurations exported: {len(scaled_feature_sets)}")
print(f"Output directory: {output_dir}")
print("Final validation and export completed successfully.")

Customers validated: 15,450
Clustering configurations exported: 7
Output directory: c:\Users\Mister\Desktop\Nova\Mestrado\DM\Projeto\Parte 2\datamining2026Preparation\data\processed
Final validation and export completed successfully.
